# Занятие 8. Деревья решений: конспект

Конспект содержит полный разбор метода с выводами формул. На семинаре
разбирают сокращенную версию — ноутбук `seminar_08.ipynb`; конспект нужен,
чтобы разобраться в деталях самостоятельно.

Содержание:

1. дерево как модель: разбиение пространства признаков, константы в листьях,
   дерево как линейная модель на индикаторах областей;
2. критерии информативности для классификации и регрессии, их свойства;
3. критерий выбора разбиения и его вычисление;
4. жадный алгоритм построения (CART) и его сложность;
5. переобучение и параметры, ограничивающие рост дерева;
6. отсечение по сложности (cost-complexity pruning);
7. важность признаков: уменьшение неоднородности и перестановочная важность;
8. свойства деревьев: инвариантность к монотонным преобразованиям
   и неустойчивость;
9. пропущенные значения;
10. разложение ошибки на смещение и разброс (bias-variance decomposition);
11. бутстрэп: доля различных объектов, оценка разброса статистики;
12. бэггинг: смещение и разброс среднего моделей, оценка вне выборки;
13. случайный лес: метод случайных подпространств, корреляция деревьев,
    важность признаков, крайне случайные деревья, сравнение границ с kNN;
14. изолирующий лес для поиска аномалий.

Обозначения: $n$ — число объектов, $p$ — число признаков, $K$ — число классов,
$x_i \in \mathbb{R}^p$ — объекты, $y_i$ — ответы, $R$ — множество объектов,
попавших в узел дерева, $|R|$ — их число.

## 0. Подготовка

Функция `show_table` раскрашивает числовые столбцы: лучшее значение ярче;
для столбцов, где лучше меньшее значение (ошибка, время), шкала обращена.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from sklearn.datasets import load_diabetes, load_wine, make_circles, make_classification, make_moons
from sklearn.ensemble import BaggingClassifier, IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, log_loss, mean_squared_error, mutual_info_score, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLASS_COLORS = ["#0072CE", "#C98A3C", "#4DAF4A"]
SEED = 42
LOWER_IS_BETTER = ("время", "ошибка", "MSE", "листьев", "откл.", "Brier")


def show_table(frame, digits=3, axis=0, lower_is_better=False):
    """Таблица с заливкой: лучшее значение ярче; для ошибки и времени лучше меньшее."""
    numeric = list(frame.select_dtypes("number").columns)
    lower = [c for c in numeric if lower_is_better or any(key in str(c) for key in LOWER_IS_BETTER)]
    higher = [c for c in numeric if c not in lower]

    styled = frame.style.format(precision=digits, subset=numeric)
    if higher:
        styled = styled.background_gradient(cmap="Greens", subset=higher, axis=axis)
    if lower:
        styled = styled.background_gradient(cmap="Greens_r", subset=lower, axis=axis)
    return styled

Используют два набора данных из sklearn:

- данные о винах (`load_wine`): 178 образцов трех сортов, 13 химических признаков;
  для рисунков берут два признака — флавоноиды и интенсивность цвета;
- данные о диабете (`load_diabetes`): 442 пациента, 10 признаков, ответ —
  количественная мера прогрессирования болезни через год; задача регрессии.

Тестовый сет — 30% объектов, со стратификацией по классу для вин.

In [ ]:
wine = load_wine(as_frame=True)
X_wine = wine.data
y_wine = wine.target.to_numpy()
FEATURES_2D = ["flavanoids", "color_intensity"]

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=SEED, stratify=y_wine
)

diabetes = load_diabetes(as_frame=True)
X_diab = diabetes.data
y_diab = diabetes.target.to_numpy()

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_diab, y_diab, test_size=0.3, random_state=SEED
)

print("вина:", Xw_train.shape, Xw_test.shape)
print("диабет:", Xd_train.shape, Xd_test.shape)

## 1. Дерево как модель

Дерево решений (decision tree) — модель, которая последовательно задает
вопросы вида «$x_j \le t$?» и в зависимости от ответа переходит к левому
или правому потомку, пока не дойдет до листа. Каждый лист $m$ соответствует
области пространства признаков $R_m$; области листьев не пересекаются
и покрывают все пространство. Предсказание — константа $c_m$ листа, в который
попал объект:

$$a(x) = \sum_{m=1}^{M} c_m\,[x \in R_m],$$

где $M$ — число листьев, $[\cdot]$ — индикатор. Каждое разбиение проводится
по одному признаку, поэтому области $R_m$ — прямоугольные параллелепипеды
со сторонами, параллельными осям координат.

Константу листа выбирают так, чтобы минимизировать функцию потерь
на обучающих объектах листа:

- **регрессия, квадратичная функция потерь.** Минимизируют
  $L(c) = \sum_{i \in R_m} (y_i - c)^2$ (среднеквадратичная ошибка, mean squared error, MSE).
  Функция выпукла, и минимум — в нуле производной: $L'(c) = -2\sum_{i \in R_m}(y_i - c) = -2\big(\sum_{i \in R_m} y_i - |R_m|\,c\big)$ обращается
  в ноль при $c = \frac{1}{|R_m|}\sum_{i \in R_m} y_i$ — среднем ответов листа;
- **регрессия, абсолютная функция потерь.** $L(c) = \sum_{i \in R_m}|y_i - c|$; ее производная
  (там, где она существует) равна $\#\{y_i < c\} - \#\{y_i > c\}$ и меняет знак в медиане,
  поэтому оптимальная константа — медиана ответов листа;
- **классификация.** Предсказывают самый частый класс листа, а вероятности
  классов оценивают долями: $\hat{p}_{mk} = \frac{1}{|R_m|}\sum_{i \in R_m}[y_i = k]$.

Пример — дерево с четырьмя листьями на двух признаках вин. Число листьев
ограничивает параметр `max_leaf_nodes=4`; при нем sklearn каждый раз делит
лист с наибольшим выигрышем от разбиения (раздел 3). Функция `draw_tree_rules`
рисует области листьев на плоскости признаков, схему дерева и правило
каждого листа.

In [ ]:
SUBSCRIPT = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")


def leaf_regions(model, features):
    """Листья дерева слева направо: номер узла и границы области (low, high) по каждому признаку."""
    structure = model.tree_
    leaves = []

    def walk(node, bounds):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            leaves.append((node, bounds))
            return

        name = features[structure.feature[node]]
        threshold = structure.threshold[node]
        low, high = bounds[name]

        left_bounds = {**bounds, name: (low, min(high, threshold))}     # ответ «да»: x <= threshold
        right_bounds = {**bounds, name: (max(low, threshold), high)}   # ответ «нет»: x > threshold

        walk(left_child, left_bounds)
        walk(right_child, right_bounds)

    unbounded = {name: (-np.inf, np.inf) for name in features}
    walk(0, unbounded)
    return leaves


def rule_text(bounds):
    """Конъюнкция условий пути; условия на один признак объединены в интервал."""
    conditions = []

    for name, (low, high) in bounds.items():
        has_low = np.isfinite(low)
        has_high = np.isfinite(high)

        if has_low and has_high:
            conditions.append(f"[{low:.2f} < {name} ≤ {high:.2f}]")
        elif has_high:
            conditions.append(f"[{name} ≤ {high:.2f}]")
        elif has_low:
            conditions.append(f"[{name} > {low:.2f}]")

    text = " ∧ ".join(conditions)
    return text


def draw_tree_rules(model, X, y, features=FEATURES_2D, class_names=("сорт 0", "сорт 1", "сорт 2")):
    """Области листьев, схема дерева и правила листьев на одном рисунке."""
    structure = model.tree_
    leaves = leaf_regions(model, features)

    leaf_number = {}
    leaf_class = {}
    for number, (node, _) in enumerate(leaves, start=1):
        leaf_number[node] = number
        leaf_class[node] = int(np.argmax(structure.value[node][0]))

    fig = plt.figure(figsize=(17, 10))
    grid = fig.add_gridspec(2, 2, height_ratios=[3, 1.3], width_ratios=[1, 1.2], hspace=0.15, wspace=0.08)
    ax_regions = fig.add_subplot(grid[0, 0])
    ax_tree = fig.add_subplot(grid[0, 1])
    ax_rules = fig.add_subplot(grid[1, :])

    # --- области листьев на плоскости признаков ---
    x_name, y_name = features
    x_lim = (X[x_name].min() - 0.3, X[x_name].max() + 0.3)
    y_lim = (X[y_name].min() - 0.5, X[y_name].max() + 0.5)

    for node, bounds in leaves:
        x_low = max(bounds[x_name][0], x_lim[0])
        x_high = min(bounds[x_name][1], x_lim[1])
        y_low = max(bounds[y_name][0], y_lim[0])
        y_high = min(bounds[y_name][1], y_lim[1])
        color = CLASS_COLORS[leaf_class[node]]

        rectangle = plt.Rectangle((x_low, y_low), x_high - x_low, y_high - y_low,
                                  facecolor=color, alpha=0.22, edgecolor=BLACK, lw=1.2)
        ax_regions.add_patch(rectangle)

        label = f"R{leaf_number[node]}".translate(SUBSCRIPT)
        center = ((x_low + x_high) / 2, (y_low + y_high) / 2)
        ax_regions.text(*center, label, ha="center", va="center", fontsize=16, color=BLACK, zorder=4,
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.7))

    for k, name in enumerate(class_names):
        is_class = y == k
        ax_regions.scatter(X.loc[is_class, x_name], X.loc[is_class, y_name], s=18,
                           color=CLASS_COLORS[k], edgecolor="white", lw=0.4, label=name, zorder=3)

    ax_regions.set_xlim(x_lim)
    ax_regions.set_ylim(y_lim)
    ax_regions.set_xlabel(x_name)
    ax_regions.set_ylabel(y_name)
    ax_regions.grid(False)
    ax_regions.legend(loc="upper right")

    # --- схема дерева: листья по горизонтали слева направо, узел посередине между потомками ---
    positions = {}
    next_leaf_x = iter(range(len(leaves)))

    def place(node, depth):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            positions[node] = (next(next_leaf_x), -depth)
            return

        place(left_child, depth + 1)
        place(right_child, depth + 1)
        middle_x = (positions[left_child][0] + positions[right_child][0]) / 2
        positions[node] = (middle_x, -depth)

    place(0, 0)

    for node, (x, y_pos) in positions.items():
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            counts = np.round(structure.value[node][0] * structure.n_node_samples[node]).astype(int)
            name = class_names[leaf_class[node]]
            label = f"R{leaf_number[node]}".translate(SUBSCRIPT) + f": {name}\n{counts.tolist()}"
            box = dict(boxstyle="round,pad=0.4", facecolor=CLASS_COLORS[leaf_class[node]], alpha=0.3, edgecolor=BLACK)
        else:
            feature_name = features[structure.feature[node]]
            label = f"{feature_name} ≤ {structure.threshold[node]:.2f}"
            box = dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=GREY)

            for child, answer in [(left_child, "да"), (right_child, "нет")]:
                child_x, child_y = positions[child]
                ax_tree.annotate("", xy=(child_x, child_y + 0.12), xytext=(x, y_pos - 0.12),
                                 arrowprops=dict(arrowstyle="->", color=GREY, lw=1.2))
                ax_tree.text((x + child_x) / 2, (y_pos + child_y) / 2, answer, ha="center", va="center", fontsize=10,
                             bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=GREY))

        ax_tree.text(x, y_pos, label, ha="center", va="center", fontsize=11, bbox=box)

    lowest_level = min(y_pos for _, y_pos in positions.values())
    ax_tree.set_xlim(-0.7, len(leaves) - 0.3)
    ax_tree.set_ylim(lowest_level - 0.5, 0.5)
    ax_tree.axis("off")

    # --- правила листьев ---
    ax_rules.axis("off")
    row_height = 1 / len(leaves)

    for row, (node, bounds) in enumerate(leaves):
        row_y = 1 - (row + 0.5) * row_height
        color = CLASS_COLORS[leaf_class[node]]

        badge = plt.Rectangle((0.02, row_y - 0.4 * row_height), 0.12, 0.8 * row_height,
                              transform=ax_rules.transAxes, facecolor=color, alpha=0.3, edgecolor=BLACK)
        ax_rules.add_patch(badge)
        ax_rules.text(0.08, row_y, class_names[leaf_class[node]], transform=ax_rules.transAxes,
                      ha="center", va="center", fontsize=13)

        rule = f"r{row + 1}".translate(SUBSCRIPT) + f"(x) = {rule_text(bounds)}"
        ax_rules.text(0.16, row_y, rule, transform=ax_rules.transAxes, ha="left", va="center", fontsize=14)

    plt.show()


small_tree = DecisionTreeClassifier(max_leaf_nodes=4, random_state=SEED)
small_tree.fit(Xw_train[FEATURES_2D], yw_train)

draw_tree_rules(small_tree, Xw_train, yw_train)

Правило листа $r_m(x) = [x \in R_m]$ — конъюнкция условий на пути от корня
к листу; условия на один признак объединяются в интервал. Ровно одно правило
равно 1 для каждого объекта, поэтому модель записывается как
$a(x) = \sum_m c_m\,r_m(x)$. Два листа могут предсказывать один класс:
здесь $R_1$ и $R_3$ относятся к сорту 1.

В sklearn то же дерево выводят функциями `plot_tree` и `export_text`.
В каждом узле указаны условие разбиения, неоднородность узла `gini`
(определена в разделе 2), число объектов `samples`, число объектов каждого
класса `value` и самый частый класс.

In [ ]:
class_names = [f"сорт {k}" for k in range(3)]

fig, ax = plt.subplots(figsize=(12, 7))
plot_tree(small_tree, feature_names=FEATURES_2D, class_names=class_names,
          filled=False, rounded=True, impurity=True, proportion=False, precision=2, ax=ax, fontsize=11)
plt.show()

print(export_text(small_tree, feature_names=FEATURES_2D))

### Дерево как линейная модель

При фиксированных областях $R_1, \dots, R_M$ дерево — линейная модель. Вводят
новые признаки — индикаторы областей $\varphi_m(x) = [x \in R_m]$ и вектор
$\varphi(x) = (\varphi_1(x), \dots, \varphi_M(x))$. Тогда

$$a(x) = \sum_{m=1}^{M} c_m\,[x \in R_m] = \langle c, \varphi(x) \rangle,$$

то есть $a$ линейна по коэффициентам $c = (c_1, \dots, c_M)$. Нелинейность дерева
целиком заключена в выборе областей, то есть в построении признаков $\varphi_m$.

**Константы листьев — решение метода наименьших квадратов.** Матрица
признаков $\Phi \in \{0, 1\}^{n \times M}$, $\Phi_{im} = [x_i \in R_m]$. Функция потерь линейной регрессии
без свободного члена

$$L(c) = \sum_{i=1}^{n}\big(y_i - \langle c, \varphi(x_i)\rangle\big)^2 = \|y - \Phi c\|^2,$$

ее минимум удовлетворяет нормальному уравнению $\Phi^\top \Phi\, c = \Phi^\top y$. Области
не пересекаются и покрывают пространство, поэтому в каждой строке $\Phi$ ровно
одна единица, и

$$(\Phi^\top \Phi)_{ml} = \sum_{i=1}^{n} [x_i \in R_m]\,[x_i \in R_l] = |R_m|\,[m = l], \qquad (\Phi^\top y)_m = \sum_{i \in R_m} y_i.$$

Матрица $\Phi^\top \Phi = \operatorname{diag}(|R_1|, \dots, |R_M|)$ диагональна, и нормальное уравнение
распадается на $M$ независимых уравнений:

$$|R_m|\, c_m = \sum_{i \in R_m} y_i \;\Longrightarrow\; c_m = \frac{1}{|R_m|}\sum_{i \in R_m} y_i.$$

Получились средние ответов листьев — константы, выведенные в начале
раздела. Для классификации метод наименьших квадратов на векторах one-hot
дает доли классов листа. Следствие: листья обученного дерева можно
использовать как признаки для другой линейной модели — так устроена
схема градиентного бустинга с логистической регрессией на индикаторах
листьев (He et al., 2014), к ней возвращаются на занятии о стекинге.

Проверка на дереве регрессии глубины 3 для диабета. Метод `apply` возвращает
для каждого объекта номер листа, в который он попал; sklearn нумерует все
узлы дерева, и внутренние тоже, поэтому номера листьев идут с пропусками.
Столбец $m$ матрицы $\Phi$ — индикатор того, что объект попал в лист с номером
`leaf_ids[m]`. Линейная регрессия без свободного члена обучается на $\Phi$.

In [ ]:
linear_tree = DecisionTreeRegressor(max_depth=3, random_state=SEED)
linear_tree.fit(Xd_train, yd_train)

leaf_of_train = linear_tree.apply(Xd_train)       # (n,): номер листа каждого обучающего объекта
leaf_ids = np.unique(leaf_of_train)               # (M,): номера листьев по возрастанию


def leaf_indicators(model, X, leaves):
    """Матрица Φ размера (n, M): Φ[i, m] = 1, если объект i попал в лист leaves[m]."""
    leaf_of_object = model.apply(X)
    indicators = np.zeros((len(leaf_of_object), len(leaves)))

    for column, leaf_id in enumerate(leaves):
        in_this_leaf = leaf_of_object == leaf_id
        indicators[in_this_leaf, column] = 1
    return indicators


Phi_train = leaf_indicators(linear_tree, Xd_train, leaf_ids)
Phi_test = leaf_indicators(linear_tree, Xd_test, leaf_ids)

ols = LinearRegression(fit_intercept=False)
ols.fit(Phi_train, yd_train)

gram = Phi_train.T @ Phi_train                    # Φ^T Φ
leaf_sizes = Phi_train.sum(axis=0)                # |R_m|
leaf_means = linear_tree.tree_.value[leaf_ids, 0, 0]

coef_gap = np.abs(ols.coef_ - leaf_means).max()
prediction_gap = np.abs(ols.predict(Phi_test) - linear_tree.predict(Xd_test)).max()

print("листьев:", len(leaf_ids))
print("Φ^T Φ = diag(|R_m|):", np.allclose(gram, np.diag(leaf_sizes)))
print(f"наибольшее расхождение коэффициентов МНК и значений листьев: {coef_gap:.2e}")
print(f"наибольшее расхождение предсказаний на тестовом сете: {prediction_gap:.2e}")

**Разложение предсказания по пути.** Пусть путь объекта $x$ проходит узлы
$t_0, t_1, \dots, t_D$ от корня до листа, $c_t$ — среднее ответов узла $t$. Сумма
разностей соседних значений телескопическая:

$$a(x) = c_{t_D} = c_{t_0} + \sum_{d=1}^{D}\big(c_{t_d} - c_{t_{d-1}}\big).$$

Каждое слагаемое относится к признаку $j(t_{d-1})$, по которому разбит узел $t_{d-1}$.
Группировка слагаемых по признакам дает

$$a(x) = c_{t_0} + \sum_{j=1}^{p} \mathrm{contrib}_j(x), \qquad \mathrm{contrib}_j(x) = \sum_{d:\ j(t_{d-1}) = j}\big(c_{t_d} - c_{t_{d-1}}\big),$$

то есть предсказание — среднее по обучающей выборке плюс сумма вкладов
признаков, своих для каждого объекта (Saabas, 2014; пакет `treeinterpreter`).
Проверка суммы и вклады признаков для первого объекта тестового сета:

In [ ]:
def path_contributions(model, X):
    """Вклады признаков (n, p): предсказание равно значению корня плюс сумма вкладов."""
    structure = model.tree_
    node_values = structure.value[:, 0, 0]          # c_t — среднее ответов каждого узла
    paths = model.decision_path(X)                  # строка i — узлы на пути объекта i
    contributions = np.zeros((paths.shape[0], structure.n_features))

    for i in range(paths.shape[0]):
        path_nodes = paths.indices[paths.indptr[i]:paths.indptr[i + 1]]      # от корня к листу

        for parent, child in zip(path_nodes[:-1], path_nodes[1:]):
            feature = structure.feature[parent]
            contributions[i, feature] += node_values[child] - node_values[parent]
    return contributions


contributions = path_contributions(linear_tree, Xd_test)
root_value = linear_tree.tree_.value[0, 0, 0]
reconstructed = root_value + contributions.sum(axis=1)

reconstruction_gap = np.abs(reconstructed - linear_tree.predict(Xd_test)).max()
print(f"наибольшее расхождение суммы вкладов и предсказания: {reconstruction_gap:.2e}")

first_object = pd.Series(contributions[0], index=X_diab.columns)
first_object = first_object[first_object != 0].sort_values()
first_prediction = linear_tree.predict(Xd_test.iloc[[0]])[0]
bar_colors = [OCHRE if value < 0 else BLUE for value in first_object]

plt.figure(figsize=(8, 3))
plt.barh(first_object.index, first_object.to_numpy(), color=bar_colors)
plt.axvline(0, color=BLACK, lw=0.8)
plt.title(f"среднее {root_value:.1f} + вклады = предсказание {first_prediction:.1f}", fontsize=10)
plt.xlabel("вклад признака")
plt.show()

## 2. Критерии информативности

Дерево строят сверху вниз: в каждом узле выбирают разбиение, после которого
потомки «чище» родителя. Чистоту узла измеряют **критерием информативности**
(impurity) $H(R)$: он равен нулю, если все объекты узла имеют один ответ,
и тем больше, чем сильнее ответы различаются.

### Классификация

Пусть $p_k = \frac{1}{|R|}\sum_{i \in R}[y_i = k]$ — доля класса $k$ в узле. Используют три критерия.

**Доля ошибок** (misclassification rate): доля объектов, не принадлежащих самому
частому классу:

$$H_{\text{err}}(R) = 1 - \max_k p_k.$$

**Индекс Джини** (Gini impurity) — вероятность ошибиться, если приписать
объекту узла случайный класс с вероятностями $p_1, \dots, p_K$ (вывод — ниже):

$$H_{\text{Gini}}(R) = 1 - \sum_{k=1}^{K} p_k^2.$$

**Энтропия** (entropy) — неопределенность класса случайного объекта узла:

$$H_{\text{ent}}(R) = -\sum_{k=1}^{K} p_k \log_2 p_k.$$

Для двух классов с долей $p$ первого класса критерии равны
$\min(p, 1 - p)$, $2p(1 - p)$ и $-p\log_2 p - (1 - p)\log_2(1 - p)$. Все три равны нулю
при $p \in \{0, 1\}$ и максимальны при $p = 0.5$. На рисунке энтропия поделена на 2,
чтобы максимумы совпадали.

In [ ]:
p_grid = np.linspace(1e-6, 1 - 1e-6, 400)                      # доля первого класса

misclassification = np.minimum(p_grid, 1 - p_grid)
gini_two_classes = 2 * p_grid * (1 - p_grid)
entropy_two_classes = -p_grid * np.log2(p_grid) - (1 - p_grid) * np.log2(1 - p_grid)

criteria = {
    "доля ошибок min(p, 1 − p)": (misclassification, GREY),
    "индекс Джини 2p(1 − p)": (gini_two_classes, BLUE),
    "энтропия / 2": (entropy_two_classes / 2, OCHRE),
}

plt.figure(figsize=(8, 4.2))
for name, (values, color) in criteria.items():
    plt.plot(p_grid, values, color=color, lw=2, label=name)

plt.xlabel("доля первого класса p")
plt.ylabel("неоднородность узла")
plt.legend()
plt.show()

Доля ошибок — кусочно-линейная функция, индекс Джини и энтропия строго
вогнуты. Это различие определяет поведение критериев при выборе разбиения.

### Вывод и свойства индекса Джини

**Вывод через вероятность ошибки.** Из узла выбирают случайный объект;
его класс $Y$ равен $k$ с вероятностью $p_k$. Классификатор знает только доли
классов в узле и приписывает объекту метку $\hat{Y}$, выбранную независимо от $Y$
с теми же вероятностями: $P(\hat{Y} = k) = p_k$. Вероятность ошибки — дополнение
вероятности совпадения; совпадение — объединение несовместных событий
$\{Y = k, \hat{Y} = k\}$, а по независимости вероятность каждого из них равна
произведению:

$$P(Y \ne \hat{Y}) = 1 - P(Y = \hat{Y}) = 1 - \sum_{k=1}^{K} P(Y = k,\ \hat{Y} = k) = 1 - \sum_{k=1}^{K} P(Y = k)\,P(\hat{Y} = k) = 1 - \sum_{k=1}^{K} p_k^2.$$

Та же величина получается группировкой по истинному классу: объект класса $k$
получает неверную метку с вероятностью $1 - p_k$, поэтому

$$P(Y \ne \hat{Y}) = \sum_{k=1}^{K} p_k (1 - p_k) = \sum_k p_k - \sum_k p_k^2 = 1 - \sum_k p_k^2,$$

так как $\sum_k p_k = 1$. Равносильная формулировка: индекс Джини — вероятность
того, что два объекта, выбранные из узла случайно с возвращением, принадлежат
разным классам.

**Вывод через дисперсию индикаторов классов.** Объекту $i$ сопоставляют вектор
индикаторов классов (one-hot) $z_i = ([y_i = 1], \dots, [y_i = K])$. Среднее $k$-й координаты
по узлу — доля класса:

$$\bar{z}_k = \frac{1}{|R|}\sum_{i \in R} [y_i = k] = p_k.$$

Выборочная дисперсия $k$-й координаты; раскрытие квадрата и равенство
$[y_i = k]^2 = [y_i = k]$ дают

$$\frac{1}{|R|}\sum_{i \in R}\big([y_i = k] - p_k\big)^2 = \frac{1}{|R|}\sum_{i \in R}[y_i = k] - 2p_k \cdot \frac{1}{|R|}\sum_{i \in R}[y_i = k] + p_k^2 = p_k - 2p_k^2 + p_k^2 = p_k(1 - p_k).$$

Сумма по координатам:

$$\sum_{k=1}^{K} \frac{1}{|R|}\sum_{i \in R}\big([y_i = k] - p_k\big)^2 = \sum_{k=1}^{K} p_k (1 - p_k) = H_{\text{Gini}}(R).$$

Индекс Джини — суммарная дисперсия векторов one-hot в узле, то есть
квадратичный критерий регрессии (раздел «Регрессия» ниже), примененный
к векторам one-hot. Дерево классификации с индексом Джини выбирает те же
разбиения, что дерево регрессии с квадратичным критерием на векторах
one-hot; проверка — после свойств.

**Свойства.**

1. **Неотрицательность.** Каждое слагаемое $p_k(1 - p_k) \ge 0$ при $p_k \in [0, 1]$
   и равно нулю только при $p_k \in \{0, 1\}$; поэтому $H_{\text{Gini}} = 0$ только у чистого узла.
2. **Максимум при равных долях.** По неравенству Коши — Буняковского для векторов
   $(1, \dots, 1)$ и $(p_1, \dots, p_K)$:

   $$1 = \Big(\sum_{k=1}^{K} 1 \cdot p_k\Big)^2 \le \Big(\sum_{k=1}^{K} 1^2\Big)\Big(\sum_{k=1}^{K} p_k^2\Big) = K \sum_k p_k^2 \;\Longrightarrow\; H_{\text{Gini}} = 1 - \sum_k p_k^2 \le 1 - \frac{1}{K},$$

   равенство — при пропорциональных векторах, то есть при $p_k = 1/K$. Наибольшее
   значение — $1/2$ для двух классов и $2/3$ для трех.
3. **Два класса.** При $p_1 = p$, $p_2 = 1 - p$:
   $H_{\text{Gini}} = 1 - p^2 - (1 - p)^2 = 2p - 2p^2 = 2p(1 - p)$.
4. **Строгая вогнутость.** Матрица вторых производных $1 - \sum_k p_k^2$ по долям
   равна $-2I$ и отрицательно определена.
5. **Наименьшая квадратичная ошибка вероятностей.** Пусть лист предсказывает
   для всех своих объектов вектор вероятностей $\mathbf{q}$. Квадратичная ошибка
   вероятностей (Brier score) объекта класса $y$ равна $\sum_k (q_k - [y = k])^2$. В среднем
   по листу доля $p_k$ объектов имеет $[y = k] = 1$, остальные — $[y = k] = 0$:

   $$B(\mathbf{q}) = \sum_{k=1}^{K}\Big(p_k (q_k - 1)^2 + (1 - p_k)\, q_k^2\Big) = \sum_{k=1}^{K}\big(q_k^2 - 2 p_k q_k + p_k\big).$$

   Производная по $q_k$ равна $2q_k - 2p_k$ и обращается в ноль при $q_k = p_k$; этот вектор
   удовлетворяет ограничению $\sum_k q_k = 1$. Значение в минимуме

   $$B(\mathbf{p}) = \sum_k \big(p_k^2 - 2p_k^2 + p_k\big) = \sum_k p_k - \sum_k p_k^2 = 1 - \sum_k p_k^2 = H_{\text{Gini}}.$$

Проверка эквивалентности: дерево `DecisionTreeClassifier` с индексом Джини
и дерево `DecisionTreeRegressor` с квадратичным критерием, обученное
на векторах one-hot, на всех 13 признаках вин при глубине 3. sklearn
для нескольких выходов усредняет дисперсию по координатам, поэтому его
неоднородность узла равна $H_{\text{Gini}} / K$, а разбиения те же.

In [ ]:
one_hot_train = np.eye(3)[yw_train]              # (n, 3): строка — вектор one-hot класса объекта

gini_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
gini_tree.fit(Xw_train, yw_train)

one_hot_tree = DecisionTreeRegressor(max_depth=3, random_state=SEED)
one_hot_tree.fit(Xw_train, one_hot_train)

same_features = np.array_equal(gini_tree.tree_.feature, one_hot_tree.tree_.feature)
same_thresholds = np.allclose(gini_tree.tree_.threshold, one_hot_tree.tree_.threshold)

gini_at_root = gini_tree.tree_.impurity[0]
regression_at_root = one_hot_tree.tree_.impurity[0]
impurity_ratio = gini_at_root / regression_at_root

probability_gap = np.abs(gini_tree.predict_proba(Xw_test) - one_hot_tree.predict(Xw_test)).max()

print(f"признаки разбиений совпадают: {same_features}, пороги совпадают: {same_thresholds}")
print(f"неоднородность корня: Джини {gini_at_root:.4f}, регрессия {regression_at_root:.4f}, отношение {impurity_ratio:.4f}")
print(f"наибольшее расхождение вероятностей классов и предсказаний регрессии: {probability_gap:.2e}")

### Свойства энтропии

Энтропия узла с долями классов $\mathbf{p} = (p_1, \dots, p_K)$:

$$H(\mathbf{p}) = -\sum_{k=1}^{K} p_k \log_2 p_k, \qquad 0 \cdot \log_2 0 = 0.$$

Соглашение $0 \cdot \log_2 0 = 0$ согласовано с пределом $\lim_{p \to 0+} p \log_2 p = 0$, поэтому
классы, отсутствующие в узле, не дают вклада.

**1. Неотрицательность.** При $p_k \in [0, 1]$ выполняется $\log_2 p_k \le 0$, поэтому каждое слагаемое
$-p_k \log_2 p_k \ge 0$ и $H(\mathbf{p}) \ge 0$. Слагаемое равно нулю только при $p_k \in \{0, 1\}$; так как
$\sum_k p_k = 1$, равенство $H(\mathbf{p}) = 0$ означает, что доля одного класса равна 1 —
узел чистый.

**2. Максимум при равных долях.** Пусть $K_+$ — число классов с $p_k > 0$. Энтропию
записывают как математическое ожидание случайной величины $Z$, принимающей
значение $1/p_k$ с вероятностью $p_k$:

$$H(\mathbf{p}) = \sum_{k:\, p_k > 0} p_k \log_2 \frac{1}{p_k} = \mathbb{E} \log_2 Z.$$

Функция $\log_2$ вогнута, и по неравенству Йенсена $\mathbb{E}\log_2 Z \le \log_2 \mathbb{E} Z$:

$$H(\mathbf{p}) \le \log_2 \sum_{k:\, p_k > 0} p_k \cdot \frac{1}{p_k} = \log_2 K_+ \le \log_2 K.$$

Равенство в неравенстве Йенсена для строго вогнутой функции достигается
только при постоянной $Z$, то есть при $p_k = 1/K_+$ для всех классов с $p_k > 0$;
во втором неравенстве равенство — при $K_+ = K$. Вместе: $H(\mathbf{p}) = \log_2 K$ только
при $p_k = 1/K$ для всех $k$. Наибольшая
энтропия — 1 бит для двух классов и $\log_2 3 \approx 1.585$ бита для трех.

**3. Строгая вогнутость.** Слагаемое $\varphi(p) = -p \log_2 p$ имеет производные

$$\varphi'(p) = -\log_2 p - \frac{1}{\ln 2}, \qquad \varphi''(p) = -\frac{1}{p \ln 2} < 0 \quad \text{при } p > 0.$$

Энтропия — сумма функций отдельных долей, поэтому ее матрица вторых
производных диагональна с отрицательными элементами $-\frac{1}{p_k \ln 2}$ и отрицательно
определена: энтропия строго вогнута. На этом свойстве основан вывод
неотрицательности выигрыша в разделе 3.

**4. Смысл энтропии.** Величина $-\log_2 p_k$ — число бит, которое оптимальный код
отводит событию вероятности $p_k$ (Шеннон, 1948): редкие события кодируются
длиннее. Энтропия — среднее число бит на запись класса случайного объекта
узла: средняя длина $\bar{\ell}$ оптимального двоичного кода удовлетворяет
$H \le \bar{\ell} < H + 1$, то есть энтропия — нижняя граница среднего числа вопросов
«да или нет», нужных, чтобы узнать класс. Для долей вида $2^{-k}$ граница
достигается: для четырех равновероятных классов $H = \log_2 4 = 2$ — ровно два вопроса.

**5. Энтропия как наименьшая логистическая функция потерь.** Пусть лист
предсказывает для всех своих объектов один вектор вероятностей
$\mathbf{q} = (q_1, \dots, q_K)$, $\sum_k q_k = 1$. Средняя логистическая функция потерь (log-loss)
по объектам листа — сумма по объектам, сгруппированная по классам:

$$L(\mathbf{q}) = -\frac{1}{|R|}\sum_{i \in R} \log_2 q_{y_i} = -\sum_{k=1}^{K} p_k \log_2 q_k.$$

Функция $L(\mathbf{q})$ выпукла по $\mathbf{q}$ (сумма выпуклых $-p_k \log_2 q_k$), поэтому ее стационарная
точка на множестве $\sum_k q_k = 1$ — минимум. Ее ищут методом множителей Лагранжа:

$$\mathcal{L}(\mathbf{q}, \lambda) = -\sum_k p_k \log_2 q_k + \lambda\Big(\sum_k q_k - 1\Big), \qquad \frac{\partial \mathcal{L}}{\partial q_k} = -\frac{p_k}{q_k \ln 2} + \lambda = 0 \;\Longrightarrow\; q_k = \frac{p_k}{\lambda \ln 2}.$$

Подстановка в ограничение: $\sum_k q_k = \frac{1}{\lambda \ln 2}\sum_k p_k = \frac{1}{\lambda \ln 2} = 1$, откуда $q_k = p_k$. Значение в минимуме

$$L(\mathbf{p}) = -\sum_k p_k \log_2 p_k = H(\mathbf{p}).$$

Превышение потерь при любом другом $\mathbf{q}$ — дивергенция Кульбака — Лейблера (Kullback — Leibler
divergence; это не расстояние: она несимметрична):

$$L(\mathbf{q}) - H(\mathbf{p}) = \sum_k p_k \log_2 \frac{p_k}{q_k} = \mathrm{KL}(\mathbf{p} \,\|\, \mathbf{q}).$$

Неотрицательность следует из неравенства Йенсена для вогнутого логарифма:
$-\mathrm{KL}(\mathbf{p} \,\|\, \mathbf{q}) = \sum_k p_k \log_2 \frac{q_k}{p_k} \le \log_2 \sum_k p_k \frac{q_k}{p_k} = \log_2 \sum_k q_k = 0$, равенство — только при $\mathbf{q} = \mathbf{p}$.

Энтропия узла — наименьшая средняя логистическая функция потерь, достижимая
одним вектором вероятностей, поэтому в sklearn значения `criterion="entropy"`
и `criterion="log_loss"` задают один критерий. Индекс Джини играет ту же роль
для квадратичной ошибки вероятностей (свойство 5 индекса Джини), а дисперсия
ответов — для квадратичной ошибки константы в регрессии.

**6. Энтропия и индекс Джини.** Энтропия в натуральных логарифмах
$H_{\ln}(\mathbf{p}) = -\sum_k p_k \ln p_k = H(\mathbf{p}) \ln 2$. Из неравенства $\ln x \le x - 1$ при $x = p_k$
следует $-\ln p_k \ge 1 - p_k$; умножение на $p_k$ и суммирование по классам дают

$$H_{\ln}(\mathbf{p}) \ge \sum_k p_k (1 - p_k) = H_{\text{Gini}}(\mathbf{p}),$$

с равенством только для чистого узла. Разложение $-\ln p_k$ в ряд Тейлора около
$p_k = 1$: $-\ln p_k = (1 - p_k) + \frac{(1 - p_k)^2}{2} + \dots$ Первый член ряда дает индекс Джини,
поэтому вблизи чистых узлов критерии почти совпадают, и на практике
деревья с ними обычно различаются мало.

На рисунке — три критерия для трех классов на симплексе
$\{\mathbf{p} : p_k \ge 0,\ p_1 + p_2 + p_3 = 1\}$: вершины — чистые узлы, центр — равные доли.
Линии уровня энтропии и индекса Джини — гладкие выпуклые кривые вокруг
центра; у доли ошибок — отрезки прямых, так как на каждой из трех областей,
где фиксирован самый частый класс, она линейна.

In [ ]:
corners = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3) / 2]])        # вершины треугольника
steps = 150

grid_counts = [(i, j, steps - i - j) for i in range(steps + 1) for j in range(steps + 1 - i)]
shares = np.array(grid_counts) / steps                             # (точек, 3): доли трех классов
points = shares @ corners                                          # доли → точка внутри треугольника

safe_shares = np.where(shares > 0, shares, 1)                      # log2(1) = 0 для отсутствующих классов
entropy_values = -(shares * np.log2(safe_shares)).sum(axis=1)
gini_values = 1 - (shares ** 2).sum(axis=1)
error_values = 1 - shares.max(axis=1)

simplex_criteria = {
    "энтропия, максимум log₂3 ≈ 1.585": entropy_values,
    "индекс Джини, максимум 2/3": gini_values,
    "доля ошибок, максимум 2/3": error_values,
}
corner_labels = ["p₁ = 1", "p₂ = 1", "p₃ = 1"]
triangle = np.vstack([corners, corners[:1]])
center = corners.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

for ax, (title, values) in zip(axes, simplex_criteria.items()):
    filled = ax.tricontourf(points[:, 0], points[:, 1], values, levels=20, cmap="viridis")
    ax.tricontour(points[:, 0], points[:, 1], values, levels=8, colors="white", linewidths=0.6)
    ax.plot(triangle[:, 0], triangle[:, 1], color=BLACK, lw=1)

    for corner, label in zip(corners, corner_labels):
        offset = (0, -16) if corner[1] == 0 else (0, 8)
        ax.annotate(label, corner, textcoords="offset points", xytext=offset, ha="center", fontsize=12)

    ax.scatter(center[0], center[1], s=40, color="white", edgecolor=BLACK, zorder=3)
    ax.set_title(title, fontsize=13, pad=26)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.colorbar(filled, ax=ax, shrink=0.75)

plt.tight_layout()
plt.show()

Численная проверка свойств 5 и 6 энтропии и свойства 5 индекса Джини на корне дерева для вин с долями классов
$\mathbf{p} = (41, 50, 33)/124$: средняя логистическая функция потерь и средняя квадратичная
ошибка вектора $\mathbf{q} = \mathbf{p}$, вычисленные функциями sklearn `log_loss`
и вручную, сравниваются с энтропией и индексом Джини; затем $\mathbf{q}$ смещают
и проверяют, что потери растут на $\mathrm{KL}(\mathbf{p} \,\|\, \mathbf{q})$.

In [ ]:
root_shares = np.bincount(yw_train) / len(yw_train)              # p = (41, 50, 33) / 124
one_hot = np.eye(3)[yw_train]

entropy_root = -(root_shares * np.log2(root_shares)).sum()
gini_root = 1 - (root_shares ** 2).sum()

probability_vectors = {
    "q = p": root_shares,
    "q = (0.2, 0.5, 0.3)": np.array([0.2, 0.5, 0.3]),
}

for name, q in probability_vectors.items():
    q_matrix = np.tile(q, (len(yw_train), 1))                     # один вектор q для всех объектов листа

    log_loss_bits = log_loss(yw_train, q_matrix) / np.log(2)       # sklearn считает в натуральных логарифмах
    brier = ((q_matrix - one_hot) ** 2).sum(axis=1).mean()
    kl = (root_shares * np.log2(root_shares / q)).sum()

    print(f"{name:20s} log-loss {log_loss_bits:.4f} = H + KL = {entropy_root:.4f} + {kl:.4f};   "
          f"Brier {brier:.4f}, индекс Джини {gini_root:.4f}")

entropy_nats = entropy_root * np.log(2)
print(f"энтропия в натуральных логарифмах {entropy_nats:.4f} ≥ индекс Джини {gini_root:.4f}")

### Регрессия

Для регрессии с квадратичной функцией потерь критерий — средний квадрат
отклонения ответов от их среднего $\bar{y}_R$, то есть выборочная дисперсия:

$$H_{\text{MSE}}(R) = \frac{1}{|R|}\sum_{i \in R} (y_i - \bar{y}_R)^2.$$

Он равен средней ошибке оптимальной константы листа. С абсолютной функцией
потерь критерий — среднее абсолютное отклонение от медианы.

## 3. Критерий выбора разбиения

Разбиение $(j, t)$ делит узел $R$ на $R_l = \{x \in R : x_j \le t\}$ и $R_r = \{x \in R : x_j > t\}$.
Его качество — уменьшение неоднородности, взвешенное по размерам потомков
(information gain):

$$Q(R, j, t) = H(R) - \frac{|R_l|}{|R|} H(R_l) - \frac{|R_r|}{|R|} H(R_r).$$

Выбирают разбиение с наибольшим $Q$. Так как $H(R)$ от разбиения не зависит,
это равносильно минимизации взвешенной неоднородности потомков
$\frac{|R_l|}{|R|} H(R_l) + \frac{|R_r|}{|R|} H(R_r)$.

**Неотрицательность выигрыша для вогнутых критериев.** Обозначают $w_l = \frac{|R_l|}{|R|}$,
$w_r = \frac{|R_r|}{|R|}$ и векторы долей классов $\mathbf{p}, \mathbf{p}_l, \mathbf{p}_r$. Каждый объект узла
попадает ровно в одного потомка, поэтому

$$\mathbf{p} = w_l\,\mathbf{p}_l + w_r\,\mathbf{p}_r, \qquad w_l + w_r = 1.$$

Индекс Джини и энтропия — вогнутые функции вектора долей $\mathbf{p}$, поэтому
по неравенству Йенсена

$$H(\mathbf{p}) = H(w_l\mathbf{p}_l + w_r\mathbf{p}_r) \ge w_l H(\mathbf{p}_l) + w_r H(\mathbf{p}_r) \;\Longrightarrow\; Q \ge 0,$$

причем для строго вогнутых критериев равенство достигается только при $\mathbf{p}_l = \mathbf{p}_r$.
Любое разбиение, меняющее доли классов в потомках, дает положительный выигрыш.
Для квадратичного критерия неотрицательность видна из формулы выигрыша ниже:
$|R|\,Q = \frac{|R_l|\,|R_r|}{|R|}(\bar{y}_l - \bar{y}_r)^2 \ge 0$.

**Выигрыш по энтропии — взаимная информация.** Пусть $Y$ — класс случайного
объекта узла, $S \in \{l, r\}$ — потомок, в который он попадает, $P(S = l) = w_l$.
Условная энтропия класса при известном потомке — взвешенная энтропия потомков:

$$H(Y \mid S) = \sum_{s} P(S = s)\, H(Y \mid S = s) = w_l H(\mathbf{p}_l) + w_r H(\mathbf{p}_r),$$

поэтому выигрыш $Q_{\text{ent}} = H(Y) - H(Y \mid S)$. Взаимная информация (mutual information)
$Y$ и $S$ по определению

$$I(Y; S) = \sum_{s, k} P(s, k) \log_2 \frac{P(s, k)}{P(s)\,P(k)}.$$

Подстановка $P(s, k) = P(s)\,P(k \mid s)$ и равенство $\sum_s P(s, k) = P(k)$ дают

$$I(Y; S) = \sum_{s, k} P(s, k) \log_2 P(k \mid s) - \sum_k P(k) \log_2 P(k) = -H(Y \mid S) + H(Y) = Q_{\text{ent}}.$$

Выигрыш по энтропии — количество информации о классе, которое дает ответ
на вопрос разбиения. Из симметрии $I(Y; S) = H(S) - H(S \mid Y) \le H(S)$ следует, что бинарное
разбиение дает не больше 1 бита. Разбиение категориального признака сразу
на много ветвей, как в алгоритмах ID3 (iterative dichotomiser 3) и C4.5, может дать большой выигрыш
за счет большого $H(S)$, поэтому C4.5 делит выигрыш на $H(S)$ (gain ratio).
Проверка на корне дерева для вин: выигрыш, вычисленный по энтропиям потомков,
и взаимная информация из sklearn (`mutual_info_score` использует натуральные
логарифмы).

In [ ]:
def impurity(counts, kind):
    p = np.asarray(counts, dtype=float) / np.sum(counts)

    if kind == "доля ошибок":
        value = 1 - p.max()
    elif kind == "индекс Джини":
        value = 1 - (p ** 2).sum()
    else:
        nonzero = p[p > 0]
        value = -(nonzero * np.log2(nonzero)).sum()
    return value


def split_gain(parent, left, right, kind):
    n = np.sum(parent)
    share_left = np.sum(left) / n
    share_right = np.sum(right) / n

    gain = impurity(parent, kind) - share_left * impurity(left, kind) - share_right * impurity(right, kind)
    return gain


goes_left = (Xw_train["color_intensity"] <= 3.82).to_numpy()

root_counts = np.bincount(yw_train, minlength=3)
left_counts = np.bincount(yw_train[goes_left], minlength=3)
right_counts = np.bincount(yw_train[~goes_left], minlength=3)

gain_bits = split_gain(root_counts, left_counts, right_counts, "энтропия")
mutual_information_nats = mutual_info_score(yw_train, goes_left)      # sklearn — в натуральных логарифмах
mutual_information_bits = mutual_information_nats / np.log(2)

print(f"выигрыш по энтропии {gain_bits:.6f} бита")
print(f"взаимная информация {mutual_information_bits:.6f} бита")

**Почему не используют долю ошибок.** Она вогнута, но не строго: на участках
линейности выигрыш может быть нулевым, хотя потомки стали чище. Пример: узел
из 400 объектов класса 0 и 400 объектов класса 1 и два разбиения:

- разбиение A: потомки (300, 100) и (100, 300);
- разбиение B: потомки (200, 400) и (200, 0) — второй потомок чистый.

In [ ]:
parent_counts = (400, 400)
splits = {
    "A: (300, 100) и (100, 300)": ((300, 100), (100, 300)),
    "B: (200, 400) и (200, 0)": ((200, 400), (200, 0)),
}

gain_table = {}
for kind in ["доля ошибок", "индекс Джини", "энтропия"]:
    gains_for_kind = {}
    for name, (left, right) in splits.items():
        gains_for_kind[name] = split_gain(parent_counts, left, right, kind)
    gain_table[kind] = gains_for_kind

show_table(pd.DataFrame(gain_table), digits=4)

Доля ошибок у обоих разбиений уменьшается одинаково, на 0.25, и не различает их.
Индекс Джини и энтропия предпочитают разбиение B, создающее чистый узел:
после него этот узел больше не нужно делить. Между собой индекс Джини
и энтропия различаются мало (свойство 6 энтропии); в sklearn по умолчанию
используют индекс Джини (`criterion="gini"`) — в нем нет логарифма, — а энтропия
доступна как `criterion="entropy"` или `"log_loss"`.

**Выигрыш для квадратичного критерия.** Пусть $S(R) = \sum_{i \in R}(y_i - \bar{y}_R)^2$,
тогда $|R|\,H_{\text{MSE}}(R) = S(R)$ и $|R|\,Q = S(R) - S(R_l) - S(R_r)$. Для объекта левого потомка
$y_i - \bar{y}_R = (y_i - \bar{y}_l) + (\bar{y}_l - \bar{y}_R)$; раскрытие квадрата и равенство
$\sum_{i \in R_l}(y_i - \bar{y}_l) = 0$ дают

$$\sum_{i \in R_l}(y_i - \bar{y}_R)^2 = S(R_l) + |R_l|\,(\bar{y}_l - \bar{y}_R)^2,$$

и аналогично для правого потомка. Сумма по двум потомкам:

$$S(R) = S(R_l) + S(R_r) + |R_l|(\bar{y}_l - \bar{y}_R)^2 + |R_r|(\bar{y}_r - \bar{y}_R)^2.$$

Подстановка $\bar{y}_R = \frac{|R_l|\bar{y}_l + |R_r|\bar{y}_r}{|R|}$, откуда $\bar{y}_l - \bar{y}_R = \frac{|R_r|}{|R|}(\bar{y}_l - \bar{y}_r)$
и $\bar{y}_r - \bar{y}_R = \frac{|R_l|}{|R|}(\bar{y}_r - \bar{y}_l)$:

$$|R|\,Q = \frac{|R_l|\,|R_r|^2 + |R_r|\,|R_l|^2}{|R|^2}\,(\bar{y}_l - \bar{y}_r)^2 = \frac{|R_l|\,|R_r|}{|R|}\,(\bar{y}_l - \bar{y}_r)^2.$$

Лучшее разбиение для регрессии максимально разводит средние потомков,
с весом, который поощряет потомков близкого размера. Формула совпадает
с приростом инерции в методе Уорда из занятий 5–6.

**Выигрыш для индекса Джини.** Индекс Джини — суммарная дисперсия векторов
one-hot (раздел 2), поэтому формулу применяют к каждой координате $k$ со средними
$p_{l,k}$ и $p_{r,k}$ и суммируют; после деления на $|R|$ и обозначений $w_l = \frac{|R_l|}{|R|}$, $w_r = \frac{|R_r|}{|R|}$:

$$Q_{\text{Gini}} = \frac{|R_l|\,|R_r|}{|R|^2}\sum_{k=1}^{K}(p_{l,k} - p_{r,k})^2 = w_l\, w_r\, \|\mathbf{p}_l - \mathbf{p}_r\|^2.$$

Проверка на разбиениях A и B: $0.5 \cdot 0.5 \cdot (0.5^2 + 0.5^2) = 0.125$
и $0.75 \cdot 0.25 \cdot \big((\tfrac{2}{3})^2 + (\tfrac{2}{3})^2\big) = 0.1875 \cdot \tfrac{8}{9} \approx 0.1667$ — совпадает с таблицей.
Выигрыш велик, если потомки сильно различаются по долям классов и имеют
близкие размеры.

### Вычисление лучшего порога

Для признака $j$ объекты узла сортируют по $x_j$; разумные пороги — середины
между соседними различными значениями, их не больше $|R| - 1$. При переходе
порога через очередной объект он переходит из правого потомка в левый,
поэтому числа объектов каждого класса слева — кумулятивные суммы,
а справа — их дополнения до итога. Для квадратичного критерия так же
накапливают $\sum y_i$ и $\sum y_i^2$:

$$S(R_l) = \sum_{i \in R_l} y_i^2 - \frac{\big(\sum_{i \in R_l} y_i\big)^2}{|R_l|}.$$

Все пороги признака перебираются за один проход после сортировки:
$O(|R| \log |R|)$ на сортировку и $O(|R|\,K)$ на проход, для всех признаков —
$O\big(p\,|R|\,(\log |R| + K)\big)$.

## 4. Жадный алгоритм построения (CART)

Построение оптимального дерева — NP-трудная задача: Хьяфил и Ривест (1976)
доказали NP-полноту построения дерева с наименьшим ожидаемым числом
проверок, и для критериев ошибки задача не проще. Поэтому дерево строят жадно, алгоритмом
CART (classification and regression trees; Брейман и соавторы, 1984):

1. начинают с корня, содержащего все обучающие объекты;
2. в текущем узле перебирают все признаки и пороги и выбирают разбиение
   с наибольшим выигрышем $Q$;
3. если выполнен критерий остановки, узел становится листом с константой
   из раздела 1, иначе его делят и применяют шаги 2–3 к обоим потомкам.

Если несколько разбиений дают одинаковый наибольший выигрыш, sklearn
оставляет первое найденное, а признаки перебирает в случайном порядке,
заданном `random_state`; поэтому деревья с разным `random_state` могут
различаться, хотя неоднородность их листьев на обучающей выборке одинакова.

Критерии остановки соответствуют параметрам sklearn:

- `max_depth` — наибольшая глубина;
- `min_samples_split` — наименьшее число объектов в узле, который разрешено делить;
- `min_samples_leaf` — наименьшее число объектов в листе;
- `max_leaf_nodes` — наибольшее число листьев;
- `min_impurity_decrease` — наименьший взвешенный выигрыш $\frac{|R|}{n} Q$;
- узел чистый или все его объекты имеют одинаковые признаки.

На каждом уровне дерева узлы не пересекаются, поэтому суммарная работа
на уровне — $O(p\,n \log n)$, а на все дерево глубины $D$ — $O(p\,n \log n \cdot D)$.
Предсказание для одного объекта требует не более $D$ сравнений.

## 5. Переобучение

Без ограничений дерево растет, пока листья не станут чистыми, и запоминает
обучающую выборку: если в ней нет совпадающих объектов с разными ответами,
accuracy на ней равна 1. На рисунке — области классов
на двух признаках вин при разной глубине; в заголовках — accuracy
на обучающей выборке и на тестовом сете.

In [ ]:
def plot_regions(ax, model, X, y, title, features=FEATURES_2D, resolution=300, legend=False, X_range=None):
    """Области классов модели на плоскости двух признаков и объекты X; пределы осей — по X_range или X."""
    x_name, y_name = features
    X_range = X if X_range is None else X_range

    x_values = np.linspace(X_range[x_name].min() - 0.3, X_range[x_name].max() + 0.3, resolution)
    y_values = np.linspace(X_range[y_name].min() - 0.5, X_range[y_name].max() + 0.5, resolution)
    grid_x, grid_y = np.meshgrid(x_values, y_values)

    grid_points = pd.DataFrame({x_name: grid_x.ravel(), y_name: grid_y.ravel()})
    predicted_class = model.predict(grid_points).reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, predicted_class, levels=[-0.5, 0.5, 1.5, 2.5], colors=CLASS_COLORS, alpha=0.18)

    for k in range(3):
        is_class = y == k
        ax.scatter(X.loc[is_class, x_name], X.loc[is_class, y_name], s=16, color=CLASS_COLORS[k], label=f"сорт {k}")

    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(title, fontsize=12)
    ax.grid(False)
    if legend:
        ax.legend(loc="upper right", fontsize=10)


fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, depth in zip(axes.ravel(), [1, 2, 4, None]):
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    model.fit(Xw_train[FEATURES_2D], yw_train)

    train_accuracy = model.score(Xw_train[FEATURES_2D], yw_train)
    test_accuracy = model.score(Xw_test[FEATURES_2D], yw_test)
    n_leaves = model.get_n_leaves()

    depth_label = "без ограничений" if depth is None else f"глубина {depth}"
    title = f"{depth_label}: обучение {train_accuracy:.2f}, тест {test_accuracy:.2f}, листьев {n_leaves}"
    plot_regions(ax, model, Xw_train, yw_train, title, legend=depth == 1)

plt.tight_layout()
plt.show()

Глубокое дерево выделяет мелкие области вокруг отдельных объектов —
признак переобучения. Кривая качества от глубины на всех 13 признаках
вин: обучающая выборка, тестовый сет и кросс-валидация на обучающей выборке
(5 фолдов).

In [ ]:
curve_rows = []

for depth in range(1, 11):
    model = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    cv_accuracy = cross_val_score(model, Xw_train, yw_train, cv=5).mean()

    model.fit(Xw_train, yw_train)
    train_accuracy = model.score(Xw_train, yw_train)
    test_accuracy = model.score(Xw_test, yw_test)

    curve_rows.append({
        "глубина": depth,
        "обучение": train_accuracy,
        "кросс-валидация": cv_accuracy,
        "тест": test_accuracy,
    })

curve = pd.DataFrame(curve_rows).set_index("глубина")

plt.figure(figsize=(8, 4))
for column, color in zip(curve.columns, [GREY, OCHRE, BLUE]):
    plt.plot(curve.index, curve[column], "o-", color=color, label=column)
plt.xticks(curve.index)

plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

Регрессионное дерево на одномерных данных $y = \sin x + \varepsilon$ показывает
то же явление: предсказание — ступенчатая функция, и с ростом глубины
ступеньки начинают повторять шум.

In [ ]:
rng_1d = np.random.default_rng(0)

x_1d = np.sort(rng_1d.uniform(0, 2 * np.pi, 80))[:, None]       # (80, 1): один признак
noise_1d = rng_1d.normal(0, 0.25, len(x_1d))
y_1d = np.sin(x_1d[:, 0]) + noise_1d

x_plot = np.linspace(0, 2 * np.pi, 600)[:, None]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)

for ax, depth in zip(axes.ravel(), [1, 2, 4, 10]):
    model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    model.fit(x_1d, y_1d)

    ax.scatter(x_1d, y_1d, s=14, color=GREY, label="обучающие точки")
    ax.plot(x_plot, np.sin(x_plot), color=OCHRE, lw=1.5, ls="--", label="sin x")
    ax.plot(x_plot, model.predict(x_plot), color=BLUE, lw=2, label="дерево")
    ax.set_title(f"глубина {depth}: листьев {model.get_n_leaves()}", fontsize=12)

for ax in axes[1]:
    ax.set_xlabel("x")
for ax in axes[:, 0]:
    ax.set_ylabel("y")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

## 6. Отсечение по сложности (cost-complexity pruning)

Ограничения роста останавливают построение заранее и могут не дать
сделать полезное разбиение, если перед ним идет малоинформативное.
Альтернатива — построить полное дерево и затем удалить лишние поддеревья.

Пусть $R(T)$ — суммарная взвешенная неоднородность листьев дерева $T$
по критерию построения дерева (в sklearn — индекс Джини или энтропия, взвешенные
долями объектов; в описании Бреймана — также доля ошибок),
$|T|$ — число листьев. Для параметра $\alpha \ge 0$ минимизируют

$$R_\alpha(T) = R(T) + \alpha\,|T|.$$

При $\alpha = 0$ оптимально полное дерево, при большом $\alpha$ — один корень.

**Слабейшее звено** (weakest link). Для внутреннего узла $t$ сравнивают
поддерево $T_t$ с корнем в $t$ и лист, полученный его схлопыванием:

$$R_\alpha(T_t) = R(T_t) + \alpha\,|T_t|, \qquad R_\alpha(t) = R(t) + \alpha.$$

Схлопывание выгодно, если $R(t) + \alpha \le R(T_t) + \alpha|T_t|$, то есть

$$\alpha \ge \alpha_{\text{eff}}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}.$$

Значение $\alpha_{\text{eff}}(t)$ — выигрыш поддерева в неоднородности на каждый дополнительный
лист. Алгоритм многократно схлопывает узел с наименьшим $\alpha_{\text{eff}}$ и получает
последовательность вложенных деревьев с возрастающими порогами $\alpha$;
для каждого $\alpha$ из интервала между соседними порогами оптимально одно
дерево этой последовательности (Брейман и соавторы, 1984). В sklearn
последовательность возвращает `cost_complexity_pruning_path`, а отсечение
задает параметр `ccp_alpha`. Значение $\alpha$ выбирают кросс-валидацией.

In [ ]:
full_model = DecisionTreeClassifier(random_state=SEED)
path = full_model.cost_complexity_pruning_path(Xw_train, yw_train)
alphas = path.ccp_alphas[:-1]                    # последний порог оставляет один корень

pruning_rows = []
for alpha in alphas:
    model = DecisionTreeClassifier(random_state=SEED, ccp_alpha=alpha)
    cv_accuracy = cross_val_score(model, Xw_train, yw_train, cv=5).mean()
    model.fit(Xw_train, yw_train)

    pruning_rows.append({
        "alpha": alpha,
        "листьев": model.get_n_leaves(),
        "обучение": model.score(Xw_train, yw_train),
        "кросс-валидация": cv_accuracy,
        "тест": model.score(Xw_test, yw_test),
    })

pruning = pd.DataFrame(pruning_rows)
best_row = pruning["кросс-валидация"].idxmax()
best_alpha = pruning.loc[best_row, "alpha"]

step_positions = np.arange(len(pruning))              # шаги отсечения: пороги α почти слипаются
step_labels = [f"{alpha:.4f}\nлистьев: {leaves}" for alpha, leaves in zip(pruning["alpha"], pruning["листьев"])]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].step(pruning["alpha"], pruning["листьев"], where="post", color=BLACK)
axes[0].set_xlabel("ccp_alpha")
axes[0].set_ylabel("число листьев")

for column, color in zip(["обучение", "кросс-валидация", "тест"], [GREY, OCHRE, BLUE]):
    axes[1].plot(step_positions, pruning[column], "o-", ms=5, color=color, label=column)
axes[1].axvline(best_row, color=OCHRE, ls="--", lw=1, label=f"лучшее по кросс-валидации: α = {best_alpha:.4f}")
axes[1].set_xticks(step_positions, step_labels, fontsize=9)
axes[1].set_xlabel("шаг отсечения: ccp_alpha и число листьев")
axes[1].set_ylabel("accuracy")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()

С ростом $\alpha$ число листьев уменьшается ступенями: каждая ступень —
схлопывание очередного слабейшего звена. Accuracy на обучающей выборке
падает монотонно, а на кросс-валидации сначала растет или держится,
затем падает, когда отсекаются полезные разбиения.

## 7. Важность признаков

**Уменьшение неоднородности** (mean decrease in impurity, MDI). Для каждого
внутреннего узла $t$, разбитого по признаку $j$, вычисляют взвешенный выигрыш
$\frac{|R_t|}{n}\,Q(R_t)$ и суммируют по всем узлам с этим признаком:

$$\mathrm{MDI}_j = \sum_{t:\ j(t) = j} \frac{|R_t|}{n}\Big(H(R_t) - \frac{|R_{t,l}|}{|R_t|}H(R_{t,l}) - \frac{|R_{t,r}|}{|R_t|}H(R_{t,r})\Big).$$

В sklearn значения нормируют так, чтобы их сумма равнялась 1, — атрибут
`feature_importances_`.

**Смысл MDI.** Слагаемое формулы — польза вопроса узла $t$: насколько он
уменьшил неоднородность, с весом, равным доле обучающих объектов, которых
вопрос касается. Сумма польз всех вопросов дерева — весь порядок, который
дерево навело на обучающей выборке. Нормированная MDI признака — доля этого
порядка, пришедшаяся на вопросы по признаку. Разбор по узлам дерева
с четырьмя листьями — в семинаре.

Формулу проверяют, вычислив ее по массивам дерева `tree_` и сравнив
с атрибутом.

In [ ]:
tree_depth4 = DecisionTreeClassifier(max_depth=4, random_state=SEED)
tree_depth4.fit(Xw_train, yw_train)

structure = tree_depth4.tree_
node_share = structure.weighted_n_node_samples / structure.weighted_n_node_samples[0]     # |R_t| / n
importance = np.zeros(X_wine.shape[1])

for node in range(structure.node_count):
    left_child = structure.children_left[node]
    right_child = structure.children_right[node]
    if left_child == -1:
        continue                                                 # лист: вопроса нет

    weighted_before = node_share[node] * structure.impurity[node]
    weighted_left = node_share[left_child] * structure.impurity[left_child]
    weighted_right = node_share[right_child] * structure.impurity[right_child]
    importance[structure.feature[node]] += weighted_before - weighted_left - weighted_right

importance = importance / importance.sum()
largest_gap = np.abs(importance - tree_depth4.feature_importances_).max()
print("наибольшее расхождение с feature_importances_:", largest_gap)

**Смещение MDI.** Важность вычисляется по обучающей выборке и поощряет
признаки, у которых много возможных порогов: у непрерывного признака
до $|R| - 1$ порогов в каждом узле, и среди них чаще находится порог, случайно
уменьшающий неоднородность. **Перестановочная важность** (permutation
importance) измеряет, насколько падает качество модели на тестовом сете,
если случайно переставить значения признака между объектами: перестановка
разрушает связь признака с ответом и сохраняет его распределение. Смысл:
насколько предсказания на новых данных зависят от признака. Если модель
опирается на признак, после перестановки она начинает ошибаться; если
не опирается, качество не меняется.

Проверка: к признакам диабета добавляют три непрерывных шумовых признака
$\mathcal{N}(0, 1)$ и три бинарных шумовых признака, не связанных с ответом,
и обучают дерево без ограничений. Перестановочную важность — уменьшение
коэффициента детерминации $R^2$ — усредняют по 30 перестановкам, усы
показывают стандартное отклонение.

In [ ]:
rng_noise = np.random.default_rng(SEED)

noise_columns = {}
for k in range(1, 4):
    noise_columns[f"шум_непрерывный_{k}"] = rng_noise.normal(size=len(X_diab))
for k in range(1, 4):
    noise_columns[f"шум_бинарный_{k}"] = rng_noise.integers(0, 2, len(X_diab)).astype(float)

X_noisy = X_diab.assign(**noise_columns)
Xn_train, Xn_test, yn_train, yn_test = train_test_split(X_noisy, y_diab, test_size=0.3, random_state=SEED)

noisy_tree = DecisionTreeRegressor(random_state=SEED)
noisy_tree.fit(Xn_train, yn_train)

permutation = permutation_importance(noisy_tree, Xn_test, yn_test, n_repeats=30, random_state=SEED)
importances = pd.DataFrame({
    "MDI": noisy_tree.feature_importances_,
    "перестановочная": permutation.importances_mean,
    "std": permutation.importances_std,
}, index=X_noisy.columns)
importances = importances.sort_values("MDI")

is_noise = importances.index.str.startswith("шум")
bar_colors = [OCHRE if noise else BLUE for noise in is_noise]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

axes[0].barh(importances.index, importances["MDI"], color=bar_colors)
axes[0].set_title("MDI по обучающей выборке")
axes[0].set_xlabel("доля MDI")

axes[1].barh(importances.index, importances["перестановочная"], xerr=importances["std"], color=bar_colors, capsize=3)
axes[1].set_title("перестановочная важность на тестовом сете")
axes[1].set_xlabel("уменьшение R² после перестановки")
axes[1].axvline(0, color=BLACK, lw=0.8)

color_legend = [Patch(color=BLUE, label="признак данных"), Patch(color=OCHRE, label="шумовой признак")]
axes[0].legend(handles=color_legend, loc="lower right")
for ax in axes:
    ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

is_continuous_noise = importances.index.str.startswith("шум_непрерывный")
is_binary_noise = importances.index.str.startswith("шум_бинарный")
continuous_noise_mdi = importances.loc[is_continuous_noise, "MDI"].sum()
binary_noise_mdi = importances.loc[is_binary_noise, "MDI"].sum()
test_r2 = noisy_tree.score(Xn_test, yn_test)

print(f"доля MDI у непрерывного шума: {continuous_noise_mdi:.3f}, у бинарного шума: {binary_noise_mdi:.3f}")
print(f"листьев {noisy_tree.get_n_leaves()}, R² на тестовом сете {test_r2:.3f}")

Шумовые признаки выделены оранжевым. Дерево без ограничений строит почти
по листу на объект и переобучено: $R^2$ на тестовом сете отрицательный.
Мелкие узлы оно делит в том числе по непрерывному шуму: один из таких
признаков занимает по MDI третье место, выше большинства настоящих признаков.
Бинарный шум, у которого в каждом узле один возможный порог, получает
в сумме меньшую долю. Перестановочная важность шумовых признаков в пределах
разброса равна нулю. У большинства настоящих признаков, кроме `bmi` и `s5`,
она тоже неотличима от нуля: переобученное дерево извлекает из них не больше
пользы для новых объектов, чем из шума, хотя MDI по обучающей выборке
приписывает им заметную важность.

Перестановочная важность тоже имеет ограничение: если два признака сильно
коррелированы, перестановка одного мало вредит модели, так как информацию
сохраняет второй.

## 8. Свойства деревьев

**Инвариантность к монотонным преобразованиям признаков.** Разбиение
зависит только от порядка значений признака, поэтому строго возрастающее
преобразование $g$ (логарифм, стандартизация, ранги) не меняет ни разбиения
объектов, ни предсказаний: порог $t$ для $x_j$ переходит в порог между теми же
соседними значениями $g(x_j)$. Стандартизация перед деревом не нужна.
Проверка: дерево на исходных признаках вин и на признаках после
преобразования $\log(1 + x - \min x)$, сдвига и растяжения.

In [ ]:
def monotone(frame, reference):
    """Строго возрастающее преобразование: логарифм сдвинутого признака, затем сдвиг и растяжение."""
    reference_min = reference.min()
    shifted = np.log1p(frame - reference_min + 1e-9)
    reference_mean = np.log1p(reference - reference_min + 1e-9).mean()

    transformed = (shifted - reference_mean) * 3.7
    return transformed


Xw_train_transformed = monotone(Xw_train, Xw_train)
Xw_test_transformed = monotone(Xw_test, Xw_train)

tree_raw = DecisionTreeClassifier(random_state=SEED)
tree_raw.fit(Xw_train, yw_train)

tree_transformed = DecisionTreeClassifier(random_state=SEED)
tree_transformed.fit(Xw_train_transformed, yw_train)

prediction_raw = tree_raw.predict(Xw_test)
prediction_transformed = tree_transformed.predict(Xw_test_transformed)
same = np.mean(prediction_raw == prediction_transformed)

print(f"совпадение предсказаний на тестовом сете: {same:.3f}")
print(f"листьев: {tree_raw.get_n_leaves()} и {tree_transformed.get_n_leaves()}")

Совпадение может быть неполным только у тестовых объектов, попавших точно
между близкими значениями, где после нелинейного преобразования середина
интервала смещается; разбиение обучающих объектов совпадает полностью.

**Неустойчивость.** Небольшое изменение обучающей выборки может изменить
разбиение в корне, а вслед за ним все дерево. На рисунке — деревья
без ограничения глубины, обученные на четырех бутстрэп-выборках (bootstrap)
из обучающей выборки вин: объекты выбираются случайно с возвращением,
размер выборки тот же. На каждой панели — объекты той выборки, на которой
обучено дерево.

In [ ]:
rng_boot = np.random.default_rng(1)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

for ax, number in zip(axes.ravel(), range(1, 5)):
    bootstrap_index = rng_boot.integers(0, len(Xw_train), len(Xw_train))     # номера объектов, с повторами
    X_bootstrap = Xw_train[FEATURES_2D].iloc[bootstrap_index]
    y_bootstrap = yw_train[bootstrap_index]

    model = DecisionTreeClassifier(random_state=SEED)
    model.fit(X_bootstrap, y_bootstrap)

    root_feature = FEATURES_2D[model.tree_.feature[0]]
    root_threshold = model.tree_.threshold[0]
    test_accuracy = model.score(Xw_test[FEATURES_2D], yw_test)

    title = f"выборка {number}: корень {root_feature} ≤ {root_threshold:.2f}, тест {test_accuracy:.2f}"
    plot_regions(ax, model, X_bootstrap, y_bootstrap, title, legend=number == 1, X_range=Xw_train)

plt.tight_layout()
plt.show()

Во всех четырех деревьях корень делит по `color_intensity`, но порог
меняется, а нижние разбиения и области классов заметно различаются, хотя
выборки взяты из одних данных. Количественно это явление описывает
разложение ошибки на смещение и разброс (раздел 10).

## 9. Пропущенные значения

Пусть у части объектов узла $R$ значение признака $j$ неизвестно; $V_j \subset R$ —
множество таких объектов. Для деревьев используют следующие подходы.

**1. Критерий по известным значениям.** Разбиение по признаку $j$ оценивают
только на объектах с известным значением и умножают выигрыш на их долю:

$$\tilde{Q}(R, j, t) = \frac{|R \setminus V_j|}{|R|}\,Q(R \setminus V_j, j, t).$$

Множитель уменьшает выигрыш признаков с большим числом пропусков: разбиение
по ним ничего не говорит об объектах из $V_j$. Например, признак, известный
у половины объектов узла, получает половину выигрыша, вычисленного по объектам
с известным значением. Так поступает алгоритм C4.5 (Quinlan, 1993).

**2. Спуск в оба поддерева.** После выбора разбиения объект с пропуском
отправляют в оба потомка с весами, равными долям объектов с известным
значением, попавших в каждого потомка:

$$w_l = \frac{|R_l|}{|R_l| + |R_r|}, \qquad w_r = \frac{|R_r|}{|R_l| + |R_r|}, \qquad w_l + w_r = 1,$$

и дальше критерии в потомках считают с весами объектов. При предсказании
объект с пропуском в признаке узла спускается в оба поддерева, и ответ узла —
взвешенное среднее ответов поддеревьев:

$$a_R(x) = w_l\, a_{R_l}(x) + w_r\, a_{R_r}(x).$$

Для классификации так усредняют векторы вероятностей классов. Если признак
пропущен в нескольких узлах пути, формулу применяют рекурсивно, и ответ —
среднее листьев, достижимых из корня, с весами, равными произведениям
$w_l$ или $w_r$ вдоль пути к каждому листу; сумма весов всех таких листьев равна 1.

**3. Суррогатные разбиения** (surrogate splits). Для каждого узла после выбора
основного разбиения запоминают несколько запасных разбиений по другим
признакам, которые на обучающих объектах узла чаще всего отправляют объекты
в того же потомка, что и основное. Если основной признак пропущен, используют
первый суррогат с известным значением. Так устроен алгоритм CART в описании
Бреймана и соавторов (1984) и пакет `rpart` языка R; в sklearn суррогатов нет.

**4. Правило sklearn** (с версии 1.3, для `splitter="best"`). При поиске порога
объекты с пропуском по очереди отправляют целиком в левого и в правого потомка
и выбирают вариант с наибольшим выигрышем; выбранное направление хранится
в массиве `tree_.missing_go_to_left`. Если на обучении у признака в узле
пропусков не было, при предсказании объект с пропуском идет в потомка
с большим числом обучающих объектов.

**5. Заполнение пропусков.** Заполнение медианой стирает информацию о том,
что значение было пропущено. Константа вне диапазона значений признака
(например, $-1$ для неотрицательного признака) ее сохраняет: дерево может
отделить такие объекты одним порогом и выучить для них отдельное правило.
Тот же эффект дает бинарный признак-индикатор пропуска.

**Эксперимент 1: пропуски только при предсказании.** Дерево глубины 4 обучают
на винах без пропусков. На тестовом сете значения трех самых важных признаков
(`flavanoids`, `color_intensity`, `proline`) независимо скрывают с вероятностью $q$,
по 30 случайных масок на каждое $q$. Сравнивают три способа предсказания:
спуск в оба поддерева (подход 2, функция `predict_proba_weighted` по массивам
`tree_`), правило sklearn «в потомка с большим числом объектов» (подход 4)
и заполнение медианой по обучающей выборке. Метрики — accuracy и средняя
квадратичная ошибка вероятностей (Brier score), для которой меньше — лучше.

In [ ]:
MISSING_FEATURES = ["flavanoids", "color_intensity", "proline"]


def hide_values(frame, share, rng, columns=MISSING_FEATURES):
    """Копия таблицы, в которой значения столбцов columns заменены на NaN с вероятностью share."""
    hidden = frame.copy()
    for column in columns:
        is_hidden = rng.random(len(hidden)) < share
        hidden.loc[is_hidden, column] = np.nan
    return hidden


def predict_proba_weighted(model, X):
    """Вероятности классов: объект с пропуском спускается в оба поддерева с весами долей объектов."""
    structure = model.tree_

    def descend(node, x):
        left_child = structure.children_left[node]
        right_child = structure.children_right[node]

        if left_child == -1:
            leaf_value = structure.value[node, 0]
            probabilities = leaf_value / leaf_value.sum()
            return probabilities

        feature_value = x[structure.feature[node]]
        if np.isnan(feature_value):
            share_left = structure.weighted_n_node_samples[left_child] / structure.weighted_n_node_samples[node]
            left_answer = descend(left_child, x)
            right_answer = descend(right_child, x)
            probabilities = share_left * left_answer + (1 - share_left) * right_answer
        elif feature_value <= structure.threshold[node]:
            probabilities = descend(left_child, x)
        else:
            probabilities = descend(right_child, x)
        return probabilities

    X_values = np.asarray(X, dtype=float)
    result = np.array([descend(0, x) for x in X_values])
    return result


def brier(y, probabilities):
    """Средняя квадратичная ошибка вектора вероятностей."""
    one_hot = np.eye(probabilities.shape[1])[y]
    score = ((probabilities - one_hot) ** 2).sum(axis=1).mean()
    return score

In [ ]:
complete_tree = DecisionTreeClassifier(max_depth=4, random_state=SEED)
complete_tree.fit(Xw_train, yw_train)

median_imputer = SimpleImputer(strategy="median")
median_imputer.fit(Xw_train)

missing_rows = []
for share in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    for repeat in range(30):
        rng = np.random.default_rng(repeat)
        X_missing = hide_values(Xw_test, share, rng)
        X_imputed = pd.DataFrame(median_imputer.transform(X_missing), columns=Xw_test.columns)

        methods = {
            "спуск в оба поддерева": predict_proba_weighted(complete_tree, X_missing),
            "правило sklearn": complete_tree.predict_proba(X_missing),
            "заполнение медианой": complete_tree.predict_proba(X_imputed),
        }

        for method, probabilities in methods.items():
            predicted = probabilities.argmax(axis=1)
            missing_rows.append({
                "доля пропусков": share,
                "способ": method,
                "accuracy": accuracy_score(yw_test, predicted),
                "Brier": brier(yw_test, probabilities),
            })

missing_long = pd.DataFrame(missing_rows)
missing_curve = missing_long.groupby(["доля пропусков", "способ"]).mean().unstack("способ")
method_colors = {"спуск в оба поддерева": BLUE, "правило sklearn": OCHRE, "заполнение медианой": GREY}

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

for ax, metric in zip(axes, ["accuracy", "Brier"]):
    for method, color in method_colors.items():
        ax.plot(missing_curve.index, missing_curve[(metric, method)], "o-", color=color, label=method)
    ax.set_xlabel("доля скрытых значений q")
    ax.set_ylabel(metric)

axes[0].legend()
plt.tight_layout()
plt.show()

Заполнение медианой хуже двух других способов по Brier при любой доле
пропусков и по accuracy при $q \le 0.4$; при $q \ge 0.5$ его accuracy сравнима
со спуском в оба поддерева. Правило sklearn дает немного большую accuracy: 0.888 против 0.873 у спуска в оба
поддерева при $q = 0.2$. Спуск в оба поддерева дает заметно меньшую ошибку
вероятностей: Brier 0.180 против 0.225 при $q = 0.2$ и 0.365 против 0.532 при $q = 0.5$.
Правило sklearn отправляет объект с пропуском в одного потомка и сохраняет
уверенность листа, даже когда направление выбрано наугад; спуск в оба
поддерева усредняет ответы и дает менее уверенные вероятности там, где
информации нет. Если нужны вероятности классов, а не только метки, спуск
в оба поддерева предпочтительнее.

**Эксперимент 2: пропуски при обучении и при предсказании.** Значения тех же
трех признаков скрывают с вероятностью $q = 0.3$ и в обучающей выборке, и на
тестовом сете, 30 масок. Сравнивают встроенную обработку пропусков sklearn
(подход 4), заполнение медианой и заполнение константой $-1$ вне диапазона
значений (все три признака положительны); для отсчета — дерево на данных
без пропусков.

In [ ]:
def new_tree():
    model = DecisionTreeClassifier(max_depth=4, random_state=SEED)
    return model


strategies = {
    "встроенная обработка sklearn": lambda: new_tree(),
    "заполнение медианой": lambda: make_pipeline(SimpleImputer(strategy="median"), new_tree()),
    "заполнение константой −1": lambda: make_pipeline(SimpleImputer(strategy="constant", fill_value=-1), new_tree()),
}
complete_accuracy = complete_tree.score(Xw_test, yw_test)

train_missing_rows = []
for repeat in range(30):
    rng = np.random.default_rng(100 + repeat)
    X_train_missing = hide_values(Xw_train, 0.3, rng)
    X_test_missing = hide_values(Xw_test, 0.3, rng)

    train_missing_rows.append({"способ": "без пропусков", "accuracy": complete_accuracy})
    for name, make_model in strategies.items():
        model = make_model()
        model.fit(X_train_missing, yw_train)
        train_missing_rows.append({"способ": name, "accuracy": model.score(X_test_missing, yw_test)})

train_missing = pd.DataFrame(train_missing_rows).groupby("способ", sort=False)["accuracy"].agg(["mean", "std"])
train_missing.columns = ["accuracy, среднее", "accuracy, ст. откл."]
show_table(train_missing, digits=3)

Встроенная обработка sklearn и константа вне диапазона дают одинаковую
accuracy в пределах разброса между масками, заполнение медианой — меньшую.
Первые два способа позволяют дереву выучить, куда направлять объекты
с пропуском; медиана смешивает их с объектами, у которых значение признака
действительно среднее.


## 10. Разложение ошибки на смещение и разброс (bias-variance decomposition)

Пусть ответы порождаются моделью $y = f(x) + \varepsilon$, где $f$ — неизвестная
детерминированная функция, шум $\varepsilon$ не зависит от $x$, $\mathbb{E}\varepsilon = 0$,
$\mathbb{D}\varepsilon = \sigma_\varepsilon^2$. Обучающая выборка $D$ случайна, и модель, обученная на ней, —
случайная функция $a_D(x)$. Ошибку в точке $x$ усредняют по обучающим выборкам
и по шуму нового ответа $y$, который не зависит от $D$:

$$\mathrm{Err}(x) = \mathbb{E}_{D,\varepsilon}\big(y - a_D(x)\big)^2.$$

Обозначают среднее предсказание по обучающим выборкам $\bar{a}(x) = \mathbb{E}_D\, a_D(x)$.
В разность $y - a_D(x)$ подставляют $y = f(x) + \varepsilon$, добавляют и вычитают $\bar{a}(x)$
(аргумент $x$ далее опущен):

$$y - a_D = \varepsilon + (f - \bar{a}) + (\bar{a} - a_D).$$

Квадрат суммы трех слагаемых равен сумме их квадратов и удвоенных попарных
произведений:

$$(y - a_D)^2 = \varepsilon^2 + (f - \bar{a})^2 + (\bar{a} - a_D)^2 + 2\varepsilon(f - \bar{a}) + 2\varepsilon(\bar{a} - a_D) + 2(f - \bar{a})(\bar{a} - a_D).$$

Математическое ожидание каждого попарного произведения равно нулю:

- $\mathbb{E}\big[\varepsilon(f - \bar{a})\big] = (f - \bar{a})\,\mathbb{E}\varepsilon = 0$, так как $f(x) - \bar{a}(x)$ не случайна;
- $\mathbb{E}\big[\varepsilon(\bar{a} - a_D)\big] = \mathbb{E}\varepsilon \cdot \mathbb{E}_D(\bar{a} - a_D) = 0$, так как шум нового ответа не зависит от выборки;
- $\mathbb{E}_D\big[(f - \bar{a})(\bar{a} - a_D)\big] = (f - \bar{a})(\bar{a} - \mathbb{E}_D a_D) = 0$ по определению $\bar{a}$.

Ожидание квадрата шума $\mathbb{E}\varepsilon^2 = \mathbb{D}\varepsilon + (\mathbb{E}\varepsilon)^2 = \sigma_\varepsilon^2$. Остаются три слагаемых:

$$\boxed{\;\mathbb{E}_{D,\varepsilon}\big(y - a_D(x)\big)^2 = \underbrace{\sigma_\varepsilon^2}_{\text{шум}} + \underbrace{\big(f(x) - \mathbb{E}_D\, a_D(x)\big)^2}_{\text{смещение}^2} + \underbrace{\mathbb{E}_D\big(a_D(x) - \mathbb{E}_D\, a_D(x)\big)^2}_{\text{разброс}}\;}$$

- **шум** $\sigma_\varepsilon^2$ — неустранимая ошибка, не зависит от модели;
- **смещение** (bias) — отклонение среднего по обучающим выборкам предсказания
  от истинной функции; велико, если модель слишком проста, чтобы описать $f$;
- **разброс** (variance) — дисперсия предсказания при смене обучающей выборки;
  велик, если модель подстраивается под конкретную выборку.

Усреднение по распределению объектов $x$ дает то же разложение для средней
ошибки на новых объектах. Разложение выведено для квадратичной функции
потерь; для доли ошибок классификации аддитивного разложения такого вида
нет, существует несколько несовпадающих определений смещения и разброса
(Domingos, 2000), но качественные выводы те же.

**Численная оценка.** Слагаемые оценивают на данных $y = \sin x + \varepsilon$, $\sigma_\varepsilon = 0.25$:
генерируют 300 независимых обучающих выборок по 80 точек, обучают на каждой
дерево заданной глубины и вычисляют предсказания в 200 точках отрезка $[0, 2\pi]$.
Среднее предсказаний по выборкам оценивает $\bar{a}(x)$, их выборочная дисперсия —
разброс; смещение² и разброс затем усредняют по точкам. Для контроля ошибку
вычисляют напрямую по новым зашумленным ответам в тех же точках.

In [ ]:
def simulate_bias_variance(make_model, n_datasets=300, n_train=80, noise=0.25, seed=0):
    """Смещение², разброс и ошибка модели на синусе по многим обучающим выборкам."""
    rng = np.random.default_rng(seed)

    x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]      # (200, 1): точки, где оценивают слагаемые
    f_eval = np.sin(x_eval[:, 0])                         # (200,): истинная функция в этих точках

    predictions = np.empty((n_datasets, len(x_eval)))     # строка b — предсказания модели, обученной на выборке b
    for b in range(n_datasets):
        x_sample = rng.uniform(0, 2 * np.pi, n_train)[:, None]
        noise_sample = rng.normal(0, noise, n_train)
        y_sample = np.sin(x_sample[:, 0]) + noise_sample

        model = make_model()
        model.fit(x_sample, y_sample)
        predictions[b] = model.predict(x_eval)

    mean_prediction = predictions.mean(axis=0)            # (200,): среднее предсказание по выборкам
    bias_squared = np.mean((mean_prediction - f_eval) ** 2)
    variance = np.mean(predictions.var(axis=0))

    y_new = f_eval + rng.normal(0, noise, predictions.shape)
    direct_error = np.mean((y_new - predictions) ** 2)

    estimate = {
        "смещение²": bias_squared,
        "разброс": variance,
        "дисперсия шума": noise ** 2,
        "сумма трех слагаемых": bias_squared + variance + noise ** 2,
        "ошибка напрямую": direct_error,
    }
    return estimate, x_eval, predictions


bv_rows = []
bv_predictions = {}

for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    def make_tree():
        model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
        return model

    estimate, x_eval, predictions = simulate_bias_variance(make_tree)
    bv_rows.append({"глубина": str(depth), **estimate})
    bv_predictions[depth] = predictions

bias_variance = pd.DataFrame(bv_rows).set_index("глубина")
show_table(bias_variance, digits=4, lower_is_better=True)

Сумма трех слагаемых совпадает с ошибкой, вычисленной напрямую, с точностью
до погрешности моделирования. Ниже слева — предсказания деревьев глубины 1
на 30 обучающих выборках, справа — деревьев без ограничения глубины; черная
линия — среднее предсказание $\bar{a}(x)$ по всем 300 выборкам, пунктир — $\sin x$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, depth in zip(axes, [1, None]):
    predictions = bv_predictions[depth]
    mean_prediction = predictions.mean(axis=0)

    for number, one_dataset_curve in enumerate(predictions[:30]):
        label = "30 деревьев на разных выборках" if number == 0 else None
        ax.plot(x_eval, one_dataset_curve, color=BLUE, lw=0.7, alpha=0.35, label=label)
    ax.plot(x_eval, mean_prediction, color=BLACK, lw=2.2, label="среднее предсказание")
    ax.plot(x_eval, np.sin(x_eval), color=OCHRE, lw=2, ls="--", label="sin x")

    row = bias_variance.loc[str(depth)]
    depth_label = "без ограничений" if depth is None else f"глубина {depth}"
    ax.set_title(f"{depth_label}: смещение² {row['смещение²']:.3f}, разброс {row['разброс']:.3f}", fontsize=12)
    ax.set_xlabel("x")

axes[0].set_ylabel("предсказание дерева")
axes[0].legend()
plt.tight_layout()
plt.show()

depth_ticks = ["∞" if depth == "None" else depth for depth in bias_variance.index]

plt.figure(figsize=(9, 4.2))
for column, color in [("смещение²", BLUE), ("разброс", OCHRE), ("сумма трех слагаемых", BLACK)]:
    plt.plot(depth_ticks, bias_variance[column], "o-", color=color, label=column)
plt.axhline(0.25 ** 2, color=GREY, ls="--", label=r"дисперсия шума $\sigma_\varepsilon^2$")

plt.xlabel("max_depth (∞ — без ограничений)")
plt.ylabel("слагаемые ошибки, среднее по x")
plt.legend()
plt.show()

Неглубокое дерево почти не меняется от выборки к выборке, но систематически
не повторяет синус: ошибка состоит из смещения. Дерево без ограничений
в среднем повторяет синус почти точно, но каждое конкретное дерево следует
за шумом своей выборки: ошибка состоит из разброса. Наименьшая ошибка
достигается при промежуточной глубине. Смещение² с ростом глубины убывает
(после глубины 5 — в пределах погрешности моделирования), разброс в целом растет; исключение — глубина 2, где положение
ступенек сильно зависит от выборки, и разброс выше, чем при глубинах 1 и 3.

Отсюда два пути к ансамблям:

- **бэггинг** (bagging, раздел 12) и **случайный лес** (random forest,
  раздел 13) усредняют много глубоких деревьев: смещение
  не меняется, а разброс уменьшается, тем сильнее, чем слабее деревья
  коррелированы;
- **градиентный бустинг** (gradient boosting) последовательно добавляет
  неглубокие деревья, каждое из которых исправляет ошибки предыдущих,
  и уменьшает смещение.

## 11. Бутстрэп

**Бутстрэп** (bootstrap; Efron, 1979) — способ получить из одной выборки
$X^n = \{x_1, \dots, x_n\}$ много выборок того же размера: $n$ раз выбирают объект
равновероятно и с возвращением. Один объект может попасть в бутстрэп-выборку
несколько раз, а может не попасть ни разу.

**Доля объектов, не попавших в выборку.** При одном выборе объект $x_i$ не выбран
с вероятностью $1 - \frac{1}{n}$. Выборы независимы, поэтому

$$P(x_i \text{ не попал в выборку}) = \Big(1 - \frac{1}{n}\Big)^n.$$

Предел при $n \to \infty$ находят через логарифм и ряд $\ln(1 - u) = -u - \frac{u^2}{2} - \frac{u^3}{3} - \dots$
при $u = \frac{1}{n}$:

$$n \ln\Big(1 - \frac{1}{n}\Big) = n\Big(-\frac{1}{n} - \frac{1}{2n^2} - \frac{1}{3n^3} - \dots\Big) = -1 - \frac{1}{2n} - \dots \;\xrightarrow[n \to \infty]{}\; -1,$$

откуда $\big(1 - \frac{1}{n}\big)^n \to e^{-1} \approx 0.368$. Число различных объектов выборки —
сумма индикаторов $[x_i \text{ попал}]$, и по линейности математического ожидания
ожидаемая доля различных объектов

$$\frac{1}{n}\sum_{i=1}^{n} P(x_i \text{ попал}) = 1 - \Big(1 - \frac{1}{n}\Big)^n \;\xrightarrow[n \to \infty]{}\; 1 - \frac{1}{e} \approx 0.632.$$

Около 36.8% объектов в бутстрэп-выборку не попадают; их называют объектами
**вне выборки** (out-of-bag, OOB) и используют для оценки качества (ниже).
Проверка: доля различных объектов в 300 бутстрэп-выборках для разных $n$.

In [ ]:
rng_bootstrap = np.random.default_rng(SEED)
sizes = np.array([2, 3, 5, 10, 20, 50, 100, 200, 500, 1000])

unique_share = []
for n in sizes:
    shares_for_n = []
    for _ in range(300):
        bootstrap_index = rng_bootstrap.integers(0, n, n)
        n_unique = len(np.unique(bootstrap_index))
        shares_for_n.append(n_unique / n)
    unique_share.append(np.mean(shares_for_n))

theory = 1 - (1 - 1 / sizes) ** sizes
limit = 1 - np.exp(-1)

plt.figure(figsize=(8, 4))
plt.semilogx(sizes, unique_share, "o", color=BLUE, label="доля различных объектов, моделирование")
plt.semilogx(sizes, theory, color=BLACK, lw=1.5, label="1 − (1 − 1/n)ⁿ")
plt.axhline(limit, color=OCHRE, ls="--", label="1 − 1/e ≈ 0.632")
plt.xlabel("размер выборки n")
plt.ylabel("доля")
plt.legend()
plt.show()

**Бутстрэп-оценка разброса статистики.** Исходное назначение бутстрэпа —
оценить, насколько менялась бы статистика, если бы выборку собрали заново,
когда новых данных нет. Статистику вычисляют на каждой бутстрэп-выборке,
и разброс полученных значений оценивает ее разброс; 2.5% и 97.5% квантили
дают 95% доверительный интервал (percentile bootstrap). Пример: accuracy
дерева глубины 3 на тестовом сете вин из 54 объектов — бутстрэп по объектам
тестового сета, 2000 выборок.

In [ ]:
tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=SEED)
tree_depth3.fit(Xw_train, yw_train)

correct = tree_depth3.predict(Xw_test) == yw_test            # (54,): верно ли предсказан объект
n_test = len(correct)

bootstrap_accuracy = []
for _ in range(2000):
    bootstrap_index = rng_bootstrap.integers(0, n_test, n_test)
    bootstrap_accuracy.append(correct[bootstrap_index].mean())
bootstrap_accuracy = np.array(bootstrap_accuracy)

low, high = np.quantile(bootstrap_accuracy, [0.025, 0.975])
spread = bootstrap_accuracy.std()
test_accuracy = correct.mean()

first_count = int(np.floor(bootstrap_accuracy.min() * n_test))
bins = (np.arange(first_count, n_test + 2) - 0.5) / n_test      # столбец на каждое значение k / n, включая 1

plt.figure(figsize=(8, 3.8))
plt.hist(bootstrap_accuracy, bins=bins, color=BLUE)
plt.grid(axis="x", visible=False)
plt.axvline(test_accuracy, color=BLACK, lw=2, label=f"accuracy на тестовом сете {test_accuracy:.3f}")
plt.axvline(low, color=OCHRE, ls="--", label="границы 95% интервала")
plt.axvline(high, color=OCHRE, ls="--")
plt.title(f"95% бутстрэп-интервал: [{low:.3f}, {high:.3f}], стандартное отклонение {spread:.3f}", fontsize=11)
plt.xlabel("accuracy на бутстрэп-выборке тестового сета")
plt.ylabel("число бутстрэп-выборок")
plt.legend(loc="upper left")
plt.show()

Стандартное отклонение accuracy около 0.025, а 95% интервал шириной около 0.09:
тестовый сет из 54 объектов определяет accuracy с точностью около $\pm 0.05$,
и различия моделей меньше этой величины на таком тестовом сете не значимы.

## 12. Бэггинг

**Бэггинг** (bagging, bootstrap aggregating; Breiman, 1996) обучает базовый
алгоритм $\mu$ на $B$ бутстрэп-выборках $X_1, \dots, X_B$ и усредняет полученные модели
$b_k = \mu(X_k)$:

$$a(x) = \frac{1}{B}\sum_{k=1}^{B} b_k(x).$$

Для классификации усредняют векторы вероятностей классов или проводят
голосование по меткам; `BaggingClassifier` в sklearn усредняет вероятности.

**Смещение и разброс среднего.** Модели $b_k$ одинаково распределены:
случайность каждой складывается из выборки $D$ и выбора бутстрэп-выборки.
Обозначают для фиксированной точки $x$ среднее $\beta = \mathbb{E}\, b_k(x)$, дисперсию
$\sigma^2 = \mathbb{D}\, b_k(x)$ и коэффициент корреляции $\rho$ двух моделей:
$\operatorname{cov}(b_k(x), b_l(x)) = \rho\sigma^2$ при $k \ne l$. Модели обучены на бутстрэп-выборках из одной
выборки $D$ и видят во многом одни и те же объекты, поэтому коррелированы: $\rho > 0$.
Величину $\rho$ не задают: это свойство базового алгоритма и данных. Ее оценивают
выборочным коэффициентом корреляции предсказаний двух моделей по многим
обучающим выборкам — так сделано в численной проверке ниже.

Среднее ансамбля по линейности математического ожидания

$$\mathbb{E}\, a(x) = \frac{1}{B}\sum_{k=1}^{B}\mathbb{E}\, b_k(x) = \beta,$$

поэтому смещение ансамбля $f(x) - \beta$ равно смещению одной модели, обученной
на бутстрэп-выборке. Дисперсия суммы — сумма всех попарных ковариаций:

$$\mathbb{D}\, a(x) = \frac{1}{B^2}\,\mathbb{D}\Big(\sum_{k=1}^{B} b_k(x)\Big) = \frac{1}{B^2}\Big(\sum_{k=1}^{B}\mathbb{D}\, b_k(x) + \sum_{k \ne l}\operatorname{cov}\big(b_k(x), b_l(x)\big)\Big).$$

Первая сумма содержит $B$ слагаемых $\sigma^2$, вторая — $B(B - 1)$ слагаемых $\rho\sigma^2$:

$$\mathbb{D}\, a(x) = \frac{B\sigma^2 + B(B - 1)\rho\sigma^2}{B^2} = \frac{\sigma^2}{B} + \frac{B - 1}{B}\,\rho\sigma^2,$$

$$\boxed{\;\mathbb{D}\, a(x) = \rho\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2\;}$$

Следствия:

- при $\rho = 0$ разброс уменьшается в $B$ раз, при $\rho = 1$ не уменьшается;
- при $B \to \infty$ разброс стремится к $\rho\sigma^2$: дальнейшее увеличение числа моделей
  не помогает, нужно уменьшать корреляцию — так устроен случайный лес;
- ожидаемая ошибка $\sigma_\varepsilon^2 + (f - \beta)^2 + \rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$ убывает с ростом $B$, поэтому
  увеличение числа моделей в бэггинге не приводит к переобучению;
- выигрыш велик для моделей с малым смещением и большим разбросом — глубоких
  деревьев; для устойчивых моделей (линейной регрессии) $\sigma^2$ мало, и бэггинг
  почти ничего не дает.

**Численная проверка формулы.** На синусе из раздела 10 для каждой из 300
выборок $D$ обучают 50 деревьев без ограничения глубины на бутстрэп-выборках
(бэггинг собран циклом по бутстрэп-выборкам, чтобы сохранить предсказания
каждого дерева). По всем деревьям оценивают $\sigma^2(x)$ и $\rho(x)\sigma^2(x)$,
затем сравнивают разброс ансамбля из первых $B$ деревьев с формулой.

In [ ]:
rng_bagging = np.random.default_rng(7)
n_datasets, n_train, n_trees, noise = 300, 80, 50, 0.25

x_eval = np.linspace(0, 2 * np.pi, 200)[:, None]
f_eval = np.sin(x_eval[:, 0])
bagged = np.empty((n_datasets, n_trees, len(x_eval)))      # bagged[d, k] — предсказания дерева k на выборке d

for d in range(n_datasets):
    x_sample = rng_bagging.uniform(0, 2 * np.pi, n_train)[:, None]
    noise_sample = rng_bagging.normal(0, noise, n_train)
    y_sample = np.sin(x_sample[:, 0]) + noise_sample

    for k in range(n_trees):
        bootstrap_index = rng_bagging.integers(0, n_train, n_train)
        tree = DecisionTreeRegressor(random_state=SEED)
        tree.fit(x_sample[bootstrap_index], y_sample[bootstrap_index])
        bagged[d, k] = tree.predict(x_eval)

In [ ]:
beta = bagged.mean(axis=(0, 1))                  # (200,): E b_k(x)
sigma2 = bagged.var(axis=(0, 1))                 # (200,): D b_k(x)
centered = bagged - beta

first_tree = centered[:, :1]                     # (300, 1, 200): первое дерево на каждой выборке
other_trees = centered[:, 1:]                    # (300, 49, 200): остальные деревья на тех же выборках
pair_cov = (first_tree * other_trees).mean(axis=(0, 1))      # cov(b_1, b_l), l > 1
rho = pair_cov / sigma2

bagging_rows = []
for B in [1, 2, 5, 10, 20, 50]:
    ensemble = bagged[:, :B].mean(axis=1)        # (300, 200): ансамбль из первых B деревьев на каждой выборке

    bias_squared = np.mean((ensemble.mean(axis=0) - f_eval) ** 2)
    simulated_variance = np.mean(ensemble.var(axis=0))
    formula_variance = np.mean(rho * sigma2 + (1 - rho) * sigma2 / B)

    bagging_rows.append({
        "B": B,
        "смещение²": bias_squared,
        "разброс, моделирование": simulated_variance,
        "разброс, формула": formula_variance,
    })

bagging_table = pd.DataFrame(bagging_rows).set_index("B")
variance_limit = np.mean(rho * sigma2)

print(f"средние по точкам: σ² = {sigma2.mean():.4f}, ρ = {rho.mean():.3f}, предел ρσ² = {variance_limit:.4f}")
show_table(bagging_table, digits=4, lower_is_better=True)

In [ ]:
B_grid = np.arange(1, 51)
formula_curve = [np.mean(rho * sigma2 + (1 - rho) * sigma2 / B) for B in B_grid]

plt.figure(figsize=(8, 4))
plt.plot(bagging_table.index, bagging_table["разброс, моделирование"], "o", color=OCHRE, ms=8, label="разброс, моделирование")
plt.plot(B_grid, formula_curve, color=OCHRE, lw=1.5, label="ρσ² + (1 − ρ)σ²/B")
plt.axhline(variance_limit, color=OCHRE, ls="--", lw=1, label="предел ρσ²")
plt.plot(bagging_table.index, bagging_table["смещение²"], "o-", color=BLUE, label="смещение²")

plt.xlabel("число деревьев B")
plt.ylabel("среднее по точкам")
plt.legend()
plt.show()

Разброс, измеренный моделированием, совпадает с формулой и с ростом $B$
приближается к пределу $\rho\sigma^2$; смещение² от $B$ не зависит в пределах погрешности
моделирования (оценка содержит слагаемое порядка разброса, деленного на 300). Смещение дерева
на бутстрэп-выборке немного больше, чем у дерева на всей выборке: различных
объектов в ней около 63%.

**Бэггинг в sklearn.** `BaggingClassifier` и `BaggingRegressor` принимают
базовую модель и число моделей `n_estimators`. Ниже — бэггинг деревьев
без ограничения глубины на винах: области классов на двух признаках.

In [ ]:
single_tree = DecisionTreeClassifier(random_state=SEED)
single_tree.fit(Xw_train[FEATURES_2D], yw_train)

bagging_2d = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED)
bagging_2d.fit(Xw_train[FEATURES_2D], yw_train)

single_accuracy = single_tree.score(Xw_test[FEATURES_2D], yw_test)
bagging_accuracy = bagging_2d.score(Xw_test[FEATURES_2D], yw_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_regions(axes[0], single_tree, Xw_train, yw_train, f"одно дерево: тест {single_accuracy:.3f}", legend=True)
plot_regions(axes[1], bagging_2d, Xw_train, yw_train, f"бэггинг 100 деревьев: тест {bagging_accuracy:.3f}")
plt.tight_layout()
plt.show()

На двух признаках бэггинг повышает accuracy на тестовом сете с 0.870 до 0.907.
Границы областей по-прежнему состоят из отрезков, параллельных осям, но ступени
мельче: ансамбль усредняет много кусочно-постоянных функций.

### Оценка вне выборки

Чтобы оценить качество модели честно, ее проверяют на объектах, которых она
не видела при обучении: на тестовом сете или кросс-валидацией. У бэггинга
такие объекты есть без дополнительных затрат. Каждое дерево $b_k$ обучено на своей бутстрэп-выборке
$X_k$, и около 37% обучающих объектов в нее не попали, как показано выше: для этого
дерева они как новые данные.

Для объекта $x_i$ обучающей выборки берут только деревья, которые его не видели,

$$K_i = \{k : x_i \notin X_k\}, \qquad |K_i| \approx 0.37\,B,$$

и усредняют их предсказания — для классификации векторы вероятностей:

$$a_{\text{OOB}}(x_i) = \frac{1}{|K_i|}\sum_{k \in K_i} b_k(x_i).$$

**Оценка вне выборки** — качество этих предсказаний на всей обучающей выборке,
для классификации — accuracy:

$$\text{OOB} = \frac{1}{n}\sum_{i=1}^{n}\big[\arg\max_j\, a_{\text{OOB}}(x_i)_j = y_i\big],$$

где $j$ пробегает классы. В отличие от кросс-валидации, ничего не нужно обучать
заново: нужные деревья уже обучены, и оценку вычисляют при обучении ансамбля
без дополнительных моделей (`oob_score=True`).

Ограничения следуют из формулы. Предсказание $a_{\text{OOB}}(x_i)$ — среднее только
$|K_i| \approx 0.37B$ деревьев, а не всех $B$, поэтому оценка немного пессимистична,
и при малом $B$ сильно: у части объектов в $K_i$ одно-два дерева. Если объект
попал во все бутстрэп-выборки, $K_i$ пусто, и объект оценить нельзя.

Расчет вручную для бэггинга из 50 деревьев. Атрибут `estimators_samples_`
хранит номера объектов бутстрэп-выборки каждого дерева.

In [ ]:
oob_model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=50, oob_score=True, random_state=SEED)
oob_model.fit(Xw_train, yw_train)

X_train_values = Xw_train.to_numpy()
n_objects = len(X_train_values)
probability_sum = np.zeros((n_objects, 3))           # сумма вероятностей от деревьев, не видевших объект
trees_not_seen = np.zeros(n_objects, dtype=int)      # |K_i| — сколько деревьев не видели объект i

for tree, sample_index in zip(oob_model.estimators_, oob_model.estimators_samples_):
    in_bootstrap = np.zeros(n_objects, dtype=bool)
    in_bootstrap[sample_index] = True                # объекты бутстрэп-выборки этого дерева
    out_of_bag = ~in_bootstrap                       # объекты, которых дерево не видело

    tree_probabilities = tree.predict_proba(X_train_values[out_of_bag])
    probability_sum[out_of_bag] += tree_probabilities
    trees_not_seen[out_of_bag] += 1

oob_probabilities = probability_sum / trees_not_seen[:, None]
oob_prediction = oob_probabilities.argmax(axis=1)
manual_oob_accuracy = np.mean(oob_prediction == yw_train)

share_not_in_bootstrap = (1 - 1 / n_objects) ** n_objects       # вероятность не попасть в бутстрэп-выборку
expected_not_seen = 50 * share_not_in_bootstrap

print(f"деревьев, не видевших объект: в среднем {trees_not_seen.mean():.1f} из 50, ожидание {expected_not_seen:.1f}")
print(f"меньше всего: {trees_not_seen.min()}, больше всего: {trees_not_seen.max()}")
print(f"оценка вне выборки вручную: {manual_oob_accuracy:.3f}")
print(f"oob_score_ из sklearn:      {oob_model.oob_score_:.3f}")

Ручной расчет совпадает с `oob_score_`. Каждый объект оценен в среднем
18 деревьями из 50, как и предсказывает вероятность $(1 - 1/124)^{124} \approx 0.366$.

Оценка вне выборки при разном числе деревьев и для сравнения accuracy
кросс-валидации и на тестовом сете:

In [ ]:
curve_rows = []

for B in [1, 2, 5, 10, 20, 50, 100, 200]:
    has_oob = B >= 10                     # при малом B у части объектов нет деревьев, где они вне выборки
    model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=has_oob, random_state=SEED)
    model.fit(Xw_train, yw_train)

    test_accuracy = model.score(Xw_test, yw_test)
    oob_accuracy = model.oob_score_ if has_oob else np.nan

    cv_model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, random_state=SEED)
    cv_accuracy = cross_val_score(cv_model, Xw_train, yw_train, cv=5).mean()

    curve_rows.append({
        "B": B,
        "тест": test_accuracy,
        "вне выборки (OOB)": oob_accuracy,
        "кросс-валидация": cv_accuracy,
    })

bagging_curve = pd.DataFrame(curve_rows).set_index("B")
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=SEED), Xw_train, yw_train, cv=5).mean()

plt.figure(figsize=(9, 4.5))
for column, color in zip(bagging_curve.columns, [BLUE, BLACK, OCHRE]):
    plt.semilogx(bagging_curve.index, bagging_curve[column], "o-", color=color, label=column)
plt.axhline(tree_cv, color=GREY, ls="--", label=f"одно дерево, кросс-валидация {tree_cv:.3f}")

plt.xticks(bagging_curve.index, bagging_curve.index)
plt.minorticks_off()
plt.xlabel("число деревьев B")
plt.ylabel("accuracy")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

show_table(bagging_curve.T, digits=3, axis=1)

Бэггинг деревьев без ограничения глубины повышает accuracy кросс-валидации
с 0.862 у одного дерева до 0.935 при 100–200 деревьях. Оценка вне выборки
при $B \ge 50$ совпадает с кросс-валидацией в пределах 0.01 и не требует
отдельного разбиения выборки; при малом $B$ она занижена (0.863 при $B = 10$):
для части объектов мало моделей, в обучение которых они не попали, и среднее
по ним неустойчиво. На тестовом сете из 54 объектов accuracy достигает 1.000
уже при $B = 20$, но по бутстрэп-интервалу из раздела 11 различия меньше
$\pm 0.05$ на нем не значимы.

## 13. Случайный лес

У бэггинга деревьев остается предел разброса $\rho\sigma^2$: деревья обучены на похожих
выборках, задают похожие вопросы и ошибаются одинаково. Чтобы уменьшить $\rho$,
в обучение добавляют вторую случайность — в выборе признаков.

**Метод случайных подпространств** (random subspace method, RSM; Ho, 1998).
Каждую модель обучают не на всех $p$ признаках, а на случайном подмножестве
из $m < p$ признаков. Это тот же прием, что в бэггинге, только случайно выбирают
столбцы таблицы, а не строки: модели, которые видят разные признаки, ошибаются
по-разному.

**Случайный лес** (random forest; Breiman, 2001) объединяет оба приема,
бэггинг по объектам и метод случайных подпространств по признакам:

1. для $k = 1, \dots, B$ строят бутстрэп-выборку $X_k$ — это бэггинг;
2. на $X_k$ растят дерево без ограничения глубины; в каждом узле заново
   случайно выбирают $m$ признаков из $p$ и ищут лучшее разбиение только по ним — это RSM;
3. ответ леса — среднее предсказаний деревьев, для классификации — среднее
   векторов вероятностей.

Отличие от исходного RSM — выбор признаков в каждом узле, а не один раз
на дерево: сильный признак недоступен в одних узлах, и там дерево задает
другой вопрос, но в других узлах может его использовать.

Параметр $m$ — `max_features`. При $m = p$ случайности в признаках нет, и лес
совпадает с бэггингом деревьев. По умолчанию в sklearn $m = \lfloor\sqrt{p}\rfloor$ для классификации
и $m = p$ для регрессии; Брейман для регрессии рекомендовал $m = p/3$.

Четыре способа на винах в одинаковых условиях, все из 300 деревьев без ограничения
глубины: бэггинг (случайные объекты), RSM (случайные 3 признака из 13 на дерево,
все объекты), бэггинг вместе с RSM на дерево и случайный лес (бэггинг и случайные
3 признака из 13 в каждом узле). Метрика — accuracy кросс-валидации на обучающей
выборке.

In [ ]:
ensembles = {
    "одно дерево": DecisionTreeClassifier(random_state=SEED),
    "бэггинг: случайные объекты": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, random_state=SEED),
    "RSM: случайные признаки на дерево": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, bootstrap=False, max_features=3, random_state=SEED),
    "бэггинг + RSM на дерево": BaggingClassifier(
        DecisionTreeClassifier(), n_estimators=300, max_features=3, random_state=SEED),
    "случайный лес: бэггинг + RSM в каждом узле": RandomForestClassifier(
        n_estimators=300, random_state=SEED),
}

variant_rows = []
for name, model in ensembles.items():
    cv_accuracy = cross_val_score(model, Xw_train, yw_train, cv=5).mean()
    variant_rows.append({"ансамбль": name, "accuracy кросс-валидации": cv_accuracy})

variant_table = pd.DataFrame(variant_rows).set_index("ансамбль")
show_table(variant_table)

Каждая из двух случайностей помогает. Бэггинг поднимает accuracy с 0.862
у одного дерева до 0.935, случайные признаки — до 0.975–0.984. При одинаковом
числе признаков $m = 3$ выбор на дерево (0.984) и в каждом узле (0.975) на винах
дают одинаковое качество в пределах шума: разница — один объект из 124.

Причину выигрыша показывает устройство самих деревьев. Для каждого ансамбля
из 200 деревьев: какой признак стоит в корне; accuracy одного дерева на объектах
вне его бутстрэп-выборки; accuracy всего ансамбля вне выборки; как часто два
дерева дают один ответ на тестовом сете (первые 50 деревьев, все пары).

In [ ]:
bagging_trees = BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, oob_score=True, random_state=SEED)
bagging_trees.fit(Xw_train, yw_train)

forest = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=SEED)
forest.fit(Xw_train, yw_train)

X_train_values = Xw_train.to_numpy()
X_test_values = Xw_test.to_numpy()
diversity_rows = []
root_shares = {}

for name, ensemble in [("бэггинг", bagging_trees), ("случайный лес", forest)]:
    trees = ensemble.estimators_
    root_features = [X_wine.columns[tree.tree_.feature[0]] for tree in trees]
    root_shares[name] = pd.Series(root_features).value_counts(normalize=True)

    tree_oob_accuracy = []                                    # каждое дерево — на объектах вне своей выборки
    for tree, sample_index in zip(trees, ensemble.estimators_samples_):
        out_of_bag = np.ones(len(X_train_values), dtype=bool)
        out_of_bag[sample_index] = False
        tree_predictions = tree.predict(X_train_values[out_of_bag])
        tree_oob_accuracy.append(np.mean(tree_predictions == yw_train[out_of_bag]))

    test_predictions = np.array([tree.predict(X_test_values) for tree in trees])     # (деревья, объекты)
    agreement = []
    for i in range(50):
        for j in range(i + 1, 50):
            same_answer = test_predictions[i] == test_predictions[j]
            agreement.append(same_answer.mean())

    diversity_rows.append({
        "ансамбль": name,
        "accuracy одного дерева вне выборки": np.mean(tree_oob_accuracy),
        "accuracy ансамбля вне выборки": ensemble.oob_score_,
        "совпадение ответов двух деревьев на тестовом сете": np.mean(agreement),
    })

diversity_table = pd.DataFrame(diversity_rows).set_index("ансамбль")
root_table = pd.DataFrame(root_shares).fillna(0).sort_values("бэггинг", ascending=False)

positions = np.arange(len(root_table))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(positions - 0.2, root_table["бэггинг"], width=0.4, color=GREY, label="бэггинг")
ax.bar(positions + 0.2, root_table["случайный лес"], width=0.4, color=BLUE, label="случайный лес")
ax.set_xticks(positions, root_table.index, rotation=45, ha="right")
ax.set_ylabel("доля деревьев")
ax.grid(axis="x", visible=False)
ax.set_title("признак, по которому задан вопрос в корне дерева")
ax.legend()
plt.show()

diversity_table.round(3)

В бэггинге 52% деревьев начинают с вопроса по `color_intensity` — самому
сильному признаку, и деревья похожи друг на друга. В лесу `color_intensity`
попадает в число трех доступных в корне признаков с вероятностью 3/13 ≈ 0.23
и стоит в корне у 15% деревьев; корни распределены почти поровну между четырьмя
признаками (`flavanoids`, `proline`, `alcohol`, `color_intensity`). Каждое дерево
леса по отдельности хуже дерева бэггинга: accuracy вне выборки 0.866 против 0.890,
ему чаще недоступен лучший признак. Но деревья леса реже дают один и тот же ответ
(0.820 против 0.866) и ошибаются по-разному, поэтому их среднее точнее:
0.976 против 0.944 вне выборки. Доля совпадающих ответов косвенно показывает
уменьшение $\rho$: лес жертвует качеством отдельного дерева ради непохожести деревьев.

**Как `max_features` меняет $\rho$.** На винах нельзя получить много независимых
обучающих выборок, поэтому $\rho$ оценивают на синтетических данных Фридмана
(Friedman, 1991): 10 признаков, равномерных на $[0, 1]$, ответ

$$y = 10\sin(\pi x_1 x_2) + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1),$$

от признаков $x_6, \dots, x_{10}$ ответ не зависит. Как в разделе 12: 200 обучающих
выборок по 500 объектов, на каждой — два дерева леса с заданным $m$. По ним
оценивают разброс одного дерева $\sigma^2$, корреляцию $\rho$, предел разброса $\rho\sigma^2$
и смещение²; ошибка бесконечного леса — шум плюс смещение² плюс $\rho\sigma^2$.

In [ ]:
def friedman_truth(X):
    """Истинная зависимость Фридмана: от признаков с шестого по десятый ответ не зависит."""
    wave = 10 * np.sin(np.pi * X[:, 0] * X[:, 1])
    parabola = 20 * (X[:, 2] - 0.5) ** 2
    linear = 10 * X[:, 3] + 5 * X[:, 4]
    value = wave + parabola + linear
    return value


rng_forest = np.random.default_rng(3)
X_eval_friedman = rng_forest.uniform(0, 1, (300, 10))        # точки, где оценивают слагаемые
f_eval_friedman = friedman_truth(X_eval_friedman)

n_friedman_datasets = 200
n_friedman_objects = 500
friedman_noise = 1.0
friedman_datasets = []
for _ in range(n_friedman_datasets):
    X_sample = rng_forest.uniform(0, 1, (n_friedman_objects, 10))
    noise_sample = rng_forest.normal(0, friedman_noise, n_friedman_objects)
    y_sample = friedman_truth(X_sample) + noise_sample
    friedman_datasets.append((X_sample, y_sample))

forest_rows = []
for m in [1, 2, 3, 5, 7, 10]:                                 # m = 10 — все признаки, то есть бэггинг
    pair_predictions = np.empty((n_friedman_datasets, 2, len(X_eval_friedman)))

    for d, (X_sample, y_sample) in enumerate(friedman_datasets):
        for k in range(2):
            bootstrap_index = rng_forest.integers(0, n_friedman_objects, n_friedman_objects)
            tree_seed = int(rng_forest.integers(1_000_000))      # своя случайность выбора признаков в узлах
            tree = DecisionTreeRegressor(max_features=m, random_state=tree_seed)
            tree.fit(X_sample[bootstrap_index], y_sample[bootstrap_index])
            pair_predictions[d, k] = tree.predict(X_eval_friedman)

    mean_prediction = pair_predictions.mean(axis=(0, 1))
    sigma2_m = pair_predictions.var(axis=(0, 1))
    centered = pair_predictions - mean_prediction
    covariance_m = (centered[:, 0] * centered[:, 1]).mean(axis=0)
    rho_m = covariance_m / sigma2_m

    bias_squared = np.mean((mean_prediction - f_eval_friedman) ** 2)
    floor = np.mean(rho_m * sigma2_m)

    forest_rows.append({
        "max_features": m,
        "разброс дерева σ²": sigma2_m.mean(),
        "корреляция ρ": rho_m.mean(),
        "предел разброса ρσ²": floor,
        "смещение²": bias_squared,
        "ошибка леса при B → ∞": friedman_noise ** 2 + bias_squared + floor,
    })

forest_table = pd.DataFrame(forest_rows).set_index("max_features")
show_table(forest_table, digits=3, lower_is_better=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(forest_table.index, forest_table["корреляция ρ"], "o-", color=BLUE)
axes[0].set_xlabel("max_features m")
axes[0].set_ylabel("ρ")
axes[0].set_title("корреляция предсказаний двух деревьев")

for column, color in [("предел разброса ρσ²", OCHRE), ("смещение²", BLUE), ("ошибка леса при B → ∞", BLACK)]:
    axes[1].plot(forest_table.index, forest_table[column], "o-", color=color, label=column)
axes[1].set_xlabel("max_features m")
axes[1].set_ylabel("средний квадрат ошибки")
axes[1].set_title("слагаемые ошибки бесконечного леса")
axes[1].legend()

for ax in axes:
    ax.set_xticks(forest_table.index)

plt.tight_layout()
plt.show()

С уменьшением $m$ деревья становятся менее похожими: $\rho \approx 0.10$ у бэггинга
($m = 10$) и 0.015 при $m = 1$, предел разброса $\rho\sigma^2$ падает с 0.84 до 0.24–0.29.
Цена — качество отдельного дерева: при малом $m$ оно чаще задает вопросы
по бесполезным признакам $x_6, \dots, x_{10}$, разброс одного дерева $\sigma^2$ растет
с 8.3 до 19.2, а смещение² — с 3.2 до 8.4. Ошибка бесконечного леса наименьшая
при $m = 7$: 4.83 против 5.08 у бэггинга, а при $m = 1$ лес хуже бэггинга почти
вдвое (9.73). На этих данных основная часть ошибки — смещение, и выигрыш
от уменьшения $\rho$ небольшой; `max_features` поэтому подбирают, например,
по оценке вне выборки.

На винах: accuracy вне выборки от числа деревьев для леса и бэггинга и от `max_features`.

In [ ]:
curve_rows = []
for B in [10, 20, 50, 100, 200, 500]:
    forest_b = RandomForestClassifier(n_estimators=B, oob_score=True, random_state=SEED)
    forest_b.fit(Xw_train, yw_train)

    bagging_b = BaggingClassifier(DecisionTreeClassifier(), n_estimators=B, oob_score=True, random_state=SEED)
    bagging_b.fit(Xw_train, yw_train)

    curve_rows.append({"B": B, "случайный лес": forest_b.oob_score_, "бэггинг": bagging_b.oob_score_})

forest_curve = pd.DataFrame(curve_rows).set_index("B")

m_rows = []
for m in [1, 2, 3, 6, 13]:
    forest_m = RandomForestClassifier(n_estimators=300, max_features=m, oob_score=True, random_state=SEED)
    forest_m.fit(Xw_train, yw_train)
    m_rows.append({"max_features": m, "accuracy вне выборки": forest_m.oob_score_})

m_table = pd.DataFrame(m_rows).set_index("max_features")

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

for column, color in [("случайный лес", BLUE), ("бэггинг", GREY)]:
    axes[0].semilogx(forest_curve.index, forest_curve[column], "o-", color=color, label=column)
axes[0].set_xticks(forest_curve.index, forest_curve.index)
axes[0].minorticks_off()
axes[0].set_xlabel("число деревьев B")
axes[0].set_ylabel("accuracy вне выборки")
axes[0].legend()

axes[1].plot(m_table.index, m_table["accuracy вне выборки"], "o-", color=BLUE)
axes[1].set_xticks(m_table.index)
axes[1].axvline(3, color=GREY, ls="--", label="по умолчанию: ⌊√13⌋ = 3")
axes[1].set_xlabel("max_features m (m = 13 — бэггинг)")
axes[1].set_ylabel("accuracy вне выборки")
axes[1].legend()

plt.tight_layout()
plt.show()

print("accuracy вне выборки от числа деревьев:")
print(forest_curve.T.round(3))
print()
print("accuracy вне выборки от max_features, 300 деревьев:")
print(m_table.T.round(3))

Лес лучше бэггинга при любом числе деревьев и выходит на 0.976 к 100
деревьям; дальнейший рост числа деревьев качество не меняет. По `max_features`
лучшие $m = 1$ и $m = 2$ (0.984), значение по умолчанию $m = 3$ дает 0.976, а $m = 13$,
то есть бэггинг, — 0.935. Значения $m$ от 1 до 6 различаются на один объект
из 124, а переход от бэггинга к лесу дает 5 объектов; в отличие от данных
Фридмана, на винах малое $m$ не ухудшает качество.

**Важность признаков в лесу** — MDI, усредненная по деревьям. Лес распределяет
важность между похожими признаками: если в узле недоступен один признак,
дерево задает вопрос по другому, который несет ту же информацию. Похожесть
признаков видна по матрице корреляций; после нее — MDI бэггинга и леса
из 200 деревьев.

In [ ]:
correlation = Xw_train.corr()
n_features = len(correlation)

fig, ax = plt.subplots(figsize=(9, 7.5))
image = ax.imshow(correlation, cmap="RdBu_r", vmin=-1, vmax=1)

for i in range(n_features):
    for j in range(n_features):
        value = correlation.iloc[i, j]
        text_color = "white" if abs(value) > 0.6 else BLACK
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8, color=text_color)

ax.set_xticks(range(n_features), correlation.columns, rotation=90)
ax.set_yticks(range(n_features), correlation.columns)
ax.grid(False)
ax.set_title("корреляция признаков вин на обучающей выборке")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()

In [ ]:
bagging_importance = np.mean([tree.feature_importances_ for tree in bagging_trees.estimators_], axis=0)

forest_importance = pd.DataFrame({
    "бэггинг": bagging_importance,
    "случайный лес": forest.feature_importances_,
}, index=X_wine.columns)
forest_importance = forest_importance.sort_values("случайный лес")

positions = np.arange(len(forest_importance))
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(positions + 0.2, forest_importance["бэггинг"], height=0.4, color=GREY, label="бэггинг, 200 деревьев")
ax.barh(positions - 0.2, forest_importance["случайный лес"], height=0.4, color=BLUE, label="случайный лес, 200 деревьев")
ax.set_yticks(positions, forest_importance.index)
ax.set_xlabel("MDI")
ax.grid(axis="y", visible=False)
ax.legend()
plt.show()

В бэггинге из тех же 200 деревьев почти вся MDI приходится на `color_intensity`
(0.300), `flavanoids` (0.290) и `proline` (0.168). В лесу важность распределена
шире: `color_intensity` 0.165, `flavanoids` 0.155, `alcohol` 0.146, `proline` 0.143,
затем `od280/od315_of_diluted_wines` 0.095, `hue` 0.094 и `total_phenols` 0.052 против 0.002
в бэггинге. Признак `flavanoids` сильно коррелирован с `total_phenols` (0.86)
и `od280/od315_of_diluted_wines` (0.77): в узлах, где `flavanoids` недоступен, деревья
спрашивают о них, и важность делится между признаками группы. Поэтому важность
в лесу показывает все полезные признаки, но каждый признак из группы
коррелированных получает только часть важности группы.

Еще сильнее деревья разнообразят **крайне случайные деревья** (extremely randomized
trees; Geurts и соавторы, 2006; `ExtraTreesClassifier`): в каждом узле для каждого
из $m$ признаков берут один случайный порог вместо лучшего. Деревья становятся
еще менее похожими и быстрее строятся.

### Границы kNN и деревьев

Метод k ближайших соседей (k nearest neighbors, kNN) с первого занятия, одно
дерево, бэггинг и случайный лес на трех наборах точек: две луны, два кольца
и классы с линейной границей, размытой шумом. Цвет фона — вероятность класса 1,
которую предсказывает модель; точки с белой обводкой — обучающая выборка,
с черной — тестовый сет; число в углу — accuracy на тестовом сете.

In [ ]:
X_linear, y_linear = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2,
                                         n_clusters_per_class=1, random_state=1)
X_linear = X_linear + 2 * np.random.default_rng(2).uniform(size=X_linear.shape)     # размывают границу
toy_datasets = {
    "луны": make_moons(n_samples=300, noise=0.3, random_state=0),
    "кольца": make_circles(n_samples=300, noise=0.2, factor=0.5, random_state=1),
    "линейная граница": (X_linear, y_linear),
}
toy_models = {
    "kNN, k = 15": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "дерево, глубина 5": lambda: DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "бэггинг, 100 деревьев": lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=SEED),
    "случайный лес, 100 деревьев": lambda: RandomForestClassifier(n_estimators=100, random_state=SEED),
}
point_colors = ListedColormap([BLUE, OCHRE])
probability_colors = LinearSegmentedColormap.from_list("класс 0 — класс 1", [BLUE, "white", OCHRE])
boundary_scores = {}

fig, axes = plt.subplots(len(toy_datasets), len(toy_models) + 1, figsize=(19, 11.5))

for row, (data_name, (X_toy, y_toy)) in enumerate(toy_datasets.items()):
    X_toy_train, X_toy_test, y_toy_train, y_toy_test = train_test_split(X_toy, y_toy, test_size=0.4, random_state=SEED)

    x_values = np.linspace(X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5, 200)
    y_values = np.linspace(X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5, 200)
    grid_x, grid_y = np.meshgrid(x_values, y_values)
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]

    test_fill = point_colors(y_toy_test)                   # RGBA цвета классов тестовых точек
    test_fill[:, 3] = 0.6                                   # прозрачна только заливка, обводка черная

    data_ax = axes[row, 0]
    data_ax.scatter(X_toy_train[:, 0], X_toy_train[:, 1], c=y_toy_train, cmap=point_colors, s=16, edgecolor="white", lw=0.5)
    data_ax.scatter(X_toy_test[:, 0], X_toy_test[:, 1], facecolors=test_fill, s=16, edgecolors=BLACK, lw=0.8)
    data_ax.set_xlim(x_values[0], x_values[-1])
    data_ax.set_ylim(y_values[0], y_values[-1])
    data_ax.set_ylabel(data_name, fontsize=15)

    for column, (model_name, make_model) in enumerate(toy_models.items(), start=1):
        model = make_model()
        model.fit(X_toy_train, y_toy_train)

        probability = model.predict_proba(grid_points)[:, 1].reshape(grid_x.shape)
        test_accuracy = model.score(X_toy_test, y_toy_test)
        boundary_scores.setdefault(model_name, {})[data_name] = test_accuracy

        ax = axes[row, column]
        extent = (x_values[0], x_values[-1], y_values[0], y_values[-1])
        background = ax.imshow(probability, extent=extent, origin="lower", aspect="auto",
                               cmap=probability_colors, vmin=0, vmax=1, alpha=0.55)
        ax.scatter(X_toy_train[:, 0], X_toy_train[:, 1], c=y_toy_train, cmap=point_colors, s=16, edgecolor="white", lw=0.5)
        ax.scatter(X_toy_test[:, 0], X_toy_test[:, 1], facecolors=test_fill, s=16, edgecolors=BLACK, lw=0.8)
        ax.text(0.97, 0.04, f"{test_accuracy:.2f}", transform=ax.transAxes, ha="right", fontsize=15,
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.85))
        if row == 0:
            ax.set_title(model_name, fontsize=15)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
axes[0, 0].set_title("данные", fontsize=15)

plt.tight_layout()
colorbar = fig.colorbar(background, ax=axes, shrink=0.5, pad=0.01)
colorbar.set_label("предсказанная вероятность класса 1", fontsize=13)
colorbar.ax.tick_params(labelsize=12)
plt.show()

boundary_table = pd.DataFrame(boundary_scores)            # строки — наборы точек, столбцы — модели
show_table(boundary_table, digits=2, axis=1)

Граница дерева состоит из отрезков, параллельных осям. Поэтому одно дерево
хуже всего на кольцах (0.78): круглую границу оно заменяет лестницей. Граница
kNN имеет произвольную форму, не привязанную к осям, и на кольцах kNN точнее
всех (0.87). Бэггинг и лес усредняют много лестниц и делают ступени мельче:
на лунах 0.96 против 0.89 у одного дерева и 0.91 у kNN, на кольцах 0.85 и 0.84.
На наборе с размытой линейной границей классы перекрываются, и все модели
ошибаются чаще (0.74–0.79); граница деревьев остается ступенчатой там, где
истинная граница — прямая. Для деревьев масштабирование признаков
не требуется: вопрос «$x_j \le t$?» не зависит от единиц измерения. kNN сравнивает
расстояния, поэтому перед ним стоит стандартизация `StandardScaler`.

## 14. Изолирующий лес

Случайные деревья применяют не только для классификации и регрессии, но и для
поиска аномалий — объектов, непохожих на остальные. Ответов здесь нет: задача
без учителя. **Изолирующий лес** (isolation forest; Liu, Ting, Zhou, 2008)
опирается на два свойства аномалий: их мало, и они далеко от основной массы.

Объект **изолируют** случайными разбиениями: выбирают случайный признак
и случайный порог между наименьшим и наибольшим значением признака в узле,
оставляют ту часть, где лежит объект, и повторяют, пока он не останется один.
Типичную точку в плотном облаке приходится отделять от многих соседей, и разбиений
нужно много. Аномалию отделяет первое же разбиение, порог которого попал
между ней и облаком, и разбиений нужно мало. Ниже — один такой процесс для самой
типичной точки облака (слева и в увеличении в центре) и для самого далекого
выброса (справа); цвет линии от темного к светлому показывает порядок разбиений.

In [ ]:
rng_iso = np.random.default_rng(0)
cluster_a = rng_iso.normal([0, 0], 0.5, (150, 2))
cluster_b = rng_iso.normal([3, 3], 0.7, (150, 2))
outliers = rng_iso.uniform(-3, 6, (20, 2))

X_iso = np.vstack([cluster_a, cluster_b, outliers])
is_outlier = np.r_[np.zeros(300, dtype=bool), np.ones(20, dtype=bool)]


def isolate(X, point_index, rng):
    """Случайные разбиения до изоляции объекта point_index; возвращает отрезки разбиений."""
    in_node = np.arange(len(X))
    box = [X[:, 0].min(), X[:, 0].max(), X[:, 1].min(), X[:, 1].max()]     # x от, x до, y от, y до
    segments = []

    while len(in_node) > 1:
        feature = rng.integers(2)                                           # случайный признак
        values = X[in_node, feature]
        low, high = values.min(), values.max()
        if low == high:
            break
        threshold = rng.uniform(low, high)                                  # случайный порог

        if feature == 0:
            segments.append(([threshold, threshold], [box[2], box[3]]))
        else:
            segments.append(([box[0], box[1]], [threshold, threshold]))

        point_goes_left = X[point_index, feature] <= threshold
        same_side = (values <= threshold) == point_goes_left
        in_node = in_node[same_side]                                        # остаются объекты с той же стороны

        if point_goes_left:
            box[2 * feature + 1] = threshold
        else:
            box[2 * feature] = threshold
    return segments


distance_to_center = np.linalg.norm(cluster_a, axis=1)
inlier_index = int(np.argmin(distance_to_center))                          # самая типичная точка

distance_to_clusters = np.minimum(np.linalg.norm(outliers, axis=1), np.linalg.norm(outliers - [3, 3], axis=1))
outlier_index = 300 + int(np.argmax(distance_to_clusters))                 # самая далекая от кластеров

def draw_isolation(ax, segments, index, title):
    """Точки, разбиения (цвет от темного к светлому — порядок) и изолируемый объект."""
    step_colors = plt.cm.viridis(np.linspace(0, 0.9, len(segments)))
    ax.scatter(X_iso[:, 0], X_iso[:, 1], s=12, color=GREY)
    for (xs, ys), color in zip(segments, step_colors):
        ax.plot(xs, ys, color=color, lw=1.1)
    ax.scatter(*X_iso[index], s=70, facecolor="none", edgecolor="red", lw=1.8, zorder=3)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("x₁")
    ax.grid(False)


rng_demo = np.random.default_rng(5)
inlier_segments = isolate(X_iso, inlier_index, rng_demo)
outlier_segments = isolate(X_iso, outlier_index, rng_demo)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), gridspec_kw={"width_ratios": [1, 0.8, 1]})

draw_isolation(axes[0], inlier_segments, inlier_index, f"обычная точка: {len(inlier_segments)} разбиений")
draw_isolation(axes[1], inlier_segments, inlier_index, "то же, увеличение около точки")
draw_isolation(axes[2], outlier_segments, outlier_index, f"аномалия: {len(outlier_segments)} разбиений")

center_x, center_y = X_iso[inlier_index]
axes[1].set_xlim(center_x - 0.4, center_x + 0.4)
axes[1].set_ylim(center_y - 0.4, center_y + 0.4)
zoom_frame = plt.Rectangle((center_x - 0.4, center_y - 0.4), 0.8, 0.8, fill=False, edgecolor=BLACK, lw=1.2)
axes[0].add_patch(zoom_frame)                               # область, показанная в увеличении
for spine in axes[1].spines.values():
    spine.set_visible(True)

axes[0].set_ylabel("x₂")
plt.tight_layout()
plt.show()

Число разбиений до изоляции $h(x)$ случайно. Распределение $h(x)$ для тех же двух
точек по 500 повторам:

In [ ]:
rng_depths = np.random.default_rng(6)
depth_samples = {"обычная точка": [], "аномалия": []}

for _ in range(500):
    inlier_segments = isolate(X_iso, inlier_index, rng_depths)
    outlier_segments = isolate(X_iso, outlier_index, rng_depths)
    depth_samples["обычная точка"].append(len(inlier_segments))
    depth_samples["аномалия"].append(len(outlier_segments))

plt.figure(figsize=(8, 3.8))
bins = np.arange(0, 30) - 0.5
for (name, depths), color in zip(depth_samples.items(), [GREY, OCHRE]):
    plt.hist(depths, bins=bins, color=color, alpha=0.7, label=f"{name}: в среднем {np.mean(depths):.1f}")
plt.xlabel("число разбиений до изоляции h(x)")
plt.ylabel("повторов")
plt.legend()
plt.show()

for name, depths in depth_samples.items():
    print(f"{name}: h(x) в среднем {np.mean(depths):.1f}, медиана {np.median(depths):.0f}")

Типичную точку облака изолируют в среднем за 14.3 разбиения, самый далекий
выброс — за 5.9. Длина пути и служит мерой аномальности: чем она короче,
тем объект необычнее.

**Алгоритм.** Изолирующий лес строит $B$ таких деревьев (по умолчанию 100), каждое
на случайной подвыборке из $\psi$ объектов без возвращения (`max_samples`, по умолчанию
$\psi = \min(256, n)$). Для объекта $x$ вычисляют среднюю по деревьям длину пути от корня
до листа $\mathbb{E}\,h(x)$ и сравнивают ее с типичной длиной пути в дереве на $\psi$ объектах:

$$c(\psi) = 2H(\psi - 1) - \frac{2(\psi - 1)}{\psi}, \qquad H(i) \approx \ln i + 0.5772,$$

где $H(i)$ — гармоническое число; $c(\psi)$ — средняя длина пути при неудачном поиске
в двоичном дереве поиска на $\psi$ объектах. **Оценка аномальности**

$$s(x) = 2^{-\mathbb{E}\,h(x) / c(\psi)}$$

близка к 1, если объект изолируют намного быстрее типичного ($\mathbb{E}\,h(x) \ll c(\psi)$),
равна 0.5 при $\mathbb{E}\,h(x) = c(\psi)$ и меньше 0.5 при более длинном пути. Подвыборка
из 256 объектов нужна, чтобы плотные скопления аномалий не маскировали друг
друга, и делает метод быстрым. В sklearn — `IsolationForest`: деревья растят
до высоты $\lceil \log_2 \psi \rceil$ (8 при $\psi = 256$), а к длине пути до листа, в котором осталось
$n_{\text{лист}}$ объектов, прибавляют $c(n_{\text{лист}})$ — ожидаемую длину оставшегося пути.
Метод `score_samples` возвращает $-s(x)$, параметр `contamination` задает ожидаемую
долю аномалий и порог для `predict`.

Качество поиска аномалий оценивают по добавленным выбросам метрикой ROC AUC
(receiver operating characteristic, area under the curve, площадь под ROC-кривой):
это вероятность того, что случайный выброс получит $s(x)$ больше, чем случайная
обычная точка.

**Проверка нормировки $c(\psi)$.** Формула заимствована из теории двоичных деревьев
поиска и для изолирующих деревьев приближенная: порог в них выбирают равномерно
между наименьшим и наибольшим значением, а не по случайному рангу. Проверка
моделированием: $n$ точек, равномерных в квадрате, средняя по точкам длина пути
до изоляции и $c(n)$.

In [ ]:
def average_path_length(n):
    """c(n) = 2H(n - 1) - 2(n - 1) / n — типичная длина пути в дереве на n объектах."""
    harmonic = np.log(n - 1) + 0.5772156649
    value = 2 * harmonic - 2 * (n - 1) / n
    return value


rng_check = np.random.default_rng(0)
path_rows = []

for n in [16, 64, 256]:
    depths = []
    for _ in range(20):
        X_uniform = rng_check.uniform(0, 1, (n, 2))
        sampled_points = rng_check.choice(n, min(n, 64), replace=False)
        for index in sampled_points:
            depths.append(len(isolate(X_uniform, index, rng_check)))

    path_rows.append({
        "n": n,
        "средняя длина пути, моделирование": np.mean(depths),
        "c(n) по формуле": average_path_length(n),
    })

pd.DataFrame(path_rows).set_index("n").round(2)

Моделирование и формула совпадают с точностью около 0.05, поэтому $c(\psi)$ годится
как типичная длина пути, с которой сравнивают $\mathbb{E}\,h(x)$.

In [ ]:
iso_forest = IsolationForest(n_estimators=200, max_samples=256, random_state=SEED)
iso_forest.fit(X_iso)

anomaly_score = -iso_forest.score_samples(X_iso)          # s(x): sklearn возвращает -s(x)
auc = roc_auc_score(is_outlier, anomaly_score)

x_values = np.linspace(-3.5, 6.5, 250)
y_values = np.linspace(-3.5, 6.5, 250)
grid_x, grid_y = np.meshgrid(x_values, y_values)
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
grid_score = -iso_forest.score_samples(grid_points).reshape(grid_x.shape)

top_20 = np.argsort(anomaly_score)[-20:]                  # 20 объектов с наибольшей оценкой

fig, ax = plt.subplots(figsize=(9, 7))
filled = ax.contourf(grid_x, grid_y, grid_score, levels=20, cmap="viridis")
ax.scatter(X_iso[~is_outlier, 0], X_iso[~is_outlier, 1], s=12, color="white", edgecolor=BLACK, lw=0.3, label="обычные точки")
ax.scatter(X_iso[is_outlier, 0], X_iso[is_outlier, 1], s=40, color=OCHRE, edgecolor=BLACK, label="добавленные выбросы")
ax.scatter(X_iso[top_20, 0], X_iso[top_20, 1], s=160, facecolor="none", edgecolor="red", lw=1.5, label="20 наибольших s(x)")
fig.colorbar(filled, ax=ax, label="оценка аномальности s(x)")
ax.set_title(f"изолирующий лес: ROC AUC по добавленным выбросам {auc:.3f}")
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.legend(loc="lower left")
ax.grid(False)
plt.show()

found = is_outlier[top_20].sum()
print(f"ROC AUC по добавленным выбросам: {auc:.3f}")
print(f"среди 20 объектов с наибольшей s(x) добавленных выбросов: {found}")
print(f"s(x): обычные точки в среднем {anomaly_score[~is_outlier].mean():.3f}, выбросы {anomaly_score[is_outlier].mean():.3f}")

Оценка $s(x)$ высока вдали от облаков и мала внутри них. По добавленным
выбросам ROC AUC 0.947; среди 20 объектов с наибольшей $s(x)$ 15 — выбросы.
Остальные выбросы случайно попали внутрь или рядом с облаками и по положению
не отличаются от обычных точек. Средняя $s(x)$ обычных точек 0.445, выбросов 0.625.

## 15. Итоги

| Что | Формула или правило |
|---|---|
| модель | $a(x) = \sum_m c_m [x \in R_m]$, области — параллелепипеды |
| константа листа | среднее (для MSE), медиана (для MAE), самый частый класс и доли классов |
| линейная модель | $a(x) = \langle c, \varphi(x) \rangle$, $\varphi_m(x) = [x \in R_m]$; МНК дает $c_m$ — средние листьев |
| критерии классификации | доля ошибок $1 - \max_k p_k$, Джини $1 - \sum_k p_k^2$, энтропия $-\sum_k p_k \log_2 p_k$ |
| критерий регрессии | дисперсия ответов узла |
| выбор разбиения | наибольший $Q = H(R) - \frac{\lvert R_l \rvert}{\lvert R \rvert}H(R_l) - \frac{\lvert R_r \rvert}{\lvert R \rvert}H(R_r)$; для MSE $\lvert R \rvert Q = \frac{\lvert R_l \rvert \lvert R_r \rvert}{\lvert R \rvert}(\bar{y}_l - \bar{y}_r)^2$ |
| построение | жадный алгоритм CART, $O(p\,n \log n)$ на уровень |
| регуляризация | `max_depth`, `min_samples_leaf`, `max_leaf_nodes`, `ccp_alpha` |
| отсечение | $R_\alpha(T) = R(T) + \alpha \lvert T \rvert$, слабейшее звено $\alpha_{\text{eff}} = \frac{R(t) - R(T_t)}{\lvert T_t \rvert - 1}$ |
| важность | MDI по обучению смещена к непрерывным признакам; перестановочная — по тестовому сету |
| свойства | инвариантность к монотонным преобразованиям; большой разброс |
| пропуски | критерий по известным значениям с множителем $\lvert R \setminus V_j \rvert / \lvert R \rvert$; спуск в оба поддерева с весами $w_l, w_r$; суррогатные разбиения; константа вне диапазона |
| разложение ошибки | $\mathbb{E}(y - a_D)^2 = \sigma^2 + \text{смещение}^2 + \text{разброс}$; глубина увеличивает разброс и уменьшает смещение |
| бутстрэп | выбор $n$ объектов с возвращением; различных около $1 - 1/e \approx 63.2\%$, остальные — вне выборки (OOB) |
| бэггинг | $a = \frac{1}{B}\sum_k b_k$; смещение как у одной модели, разброс $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$ |
| оценка вне выборки | $a_{\text{OOB}}(x_i)$ — среднее деревьев, не видевших $x_i$, около $0.37B$ деревьев |
| случайный лес | бэггинг + случайные `max_features` признаков в каждом узле; меньше $\rho$, больше смещение одного дерева |
| изолирующий лес | случайные разбиения до изоляции; $s(x) = 2^{-\mathbb{E}h(x)/c(\psi)}$, у аномалий близко к 1 |